# Towards a GIS system

In [ ]:
#| default_exp database

In [ ]:
#| export
import numpy as np
import sys
import os
import math
from math import radians, cos, sin, sqrt, atan2
import random

#data
from collections import namedtuple
from dataclasses import dataclass,  field, asdict
import json
from typing import List , Optional
from enum import Enum



In [ ]:
#| export
from dataclasses import dataclass
from typing import Optional

from datetime import datetime


In [ ]:
#| export
from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *
from fastlite import *
import fasthtml.components as fc
import httpx


In [ ]:
#| export
from importlib import resources
from pathlib import Path
import tempfile
import shutil
import sqlite3
from dataclasses import dataclass
from typing import Iterator
import numpy as np

In [ ]:
#| export
from HexMagic.climate import TerrainPatterns, DrainageBasins, Geology, TerraDemo, Terrain, GeoBounds, ClimatePreset, TerrainFactory
from HexMagic.primitives import MapCord, MapSize, MapRect, MapPath, Hex, HexGrid, HexWrapper, HexPosition, HexRegion, unique_windy_edge, HexChunk
from HexMagic.styles import StyleCSS, SVGBuilder
from HexMagic.cover import ChunkCover, TerraDemo

In [ ]:
#| export
from HexMagic.geology import River, Watershed

In [ ]:
#| export
import asyncio
from dataclasses import dataclass

In [ ]:
SVGBuilder.BUILDERHIDE = True

In [ ]:
#TerraDemo().upscaleWater(TerraDemo().maui_map())

In [ ]:
!cat ../llms.txt

## Helpers

In [ ]:
#| export
LoadResult = namedtuple('LoadResult', ['data', 'status', 'context'])
SaveResult = namedtuple('SaveResult', ['id', 'status', 'context'])

In [ ]:
#| export
@dataclass
class ChunkRef:
    """Reference to a chunk in the world - uses HexPosition for coordinates."""
    position: HexPosition  # Chunk's position in chunk-space (q+r+s=0)
    rings: int             # Hex rings per chunk
    
    @property
    def center_hex(self) -> HexPosition:
        """World hex coordinates of chunk center."""
        spacing = self.rings * 2
        return spacing * self.position
    
    @property
    def key(self) -> tuple[int, int, int]:
        """Hashable key for dict storage."""
        return (self.position.q, self.position.r, self.position.s)
    
    def __hash__(self):
        return hash(self.key)
    
    @classmethod
    def from_key(cls, key, chunk_rings: int) -> 'ChunkRef':
        if isinstance(key, tuple):
            q, r, s = key
        else:
            q, r, s = map(int, key.split(','))
        return cls(HexPosition(q, r, s), chunk_rings)




def chunk_to_world(chunk: ChunkRef, local_pos: HexPosition) -> HexPosition:
    """Map local hex position within chunk to world coordinates."""
    return chunk.center_hex + local_pos


def world_to_chunk(world_pos: HexPosition, chunk_rings: int) -> tuple[HexPosition, HexPosition]:
    """Map world position to (chunk_position, local_position_within_chunk)."""
    spacing = chunk_rings * 2
    
    chunk_q = world_pos.q // spacing
    chunk_r = world_pos.r // spacing
    chunk_s = -chunk_q - chunk_r
    
    chunk_pos = HexPosition(chunk_q, chunk_r, chunk_s)
    chunk_center = spacing * chunk_pos
    local_pos = world_pos - chunk_center
    
    return chunk_pos, local_pos


## Tables

In [ ]:
#| export
@dataclass  
class HexData:
    id: int = None
    world_id: int = 0
    q: int = 0
    r: int = 0
    s: int = 0
    grid_index: int = 0
    elevation: float = 0.0
    plate_id:int = -1
    latitude: Optional[float] = None
    longitude: Optional[float] = None
    distance_from_coast: Optional[float] = None
    watershed_id: Optional[int] = None  # NEW: FK to WatershedMeta
    chunk_q: Optional[int] = None  # Chunk origin position
    chunk_r: Optional[int] = None
    chunk_s: Optional[int] = None
    scale_level: int = 0  # 0=coarse, >0=zoomed scale factor
    modified: int = 0
    last_accessed: int = 0   # Unix timestamp
    access_count: int = 0    # Hit counter


In [ ]:
#| export
@dataclass
class HexWeather:
    """Weather data for a hex at a point in time."""
    id: int = None
    world_id: int = 0
    q: int = 0
    r: int = 0
    s: int = 0
    # Core weather fields
    temperature: float = 0.0           # °C
    precipitation: float = 0.0         # mm/year
    humidity: Optional[float] = None   # %
    # Derived/computed fields
    climate_pet: Optional[float] = None          # potential evapotranspiration
    aridity_index: Optional[float] = None
    temp_seasonality: Optional[float] = None
    precip_seasonality: Optional[float] = None
    distance_from_coast: Optional[float] = None  # hex units
    # Temporal
    season: str = ""                   # e.g., "annual", "summer", "winter"

     # NEW: Chunk identification (NULL for coarse-level weather)
    chunk_q: Optional[int] = None
    chunk_r: Optional[int] = None
    chunk_s: Optional[int] = None
    
    # NEW: For cache invalidation
    scale_level: int = 0        # 0=coarse, 1=fine chunk
    climate_name: str = ""      # e.g., "mediterranean" - invalidate if changed
    wind_dir: Optional[float] = None  # Wind direction used in computation


    modified: int = 0


In [ ]:
@dataclass
class CacheStats:
    cover_id: int
    chunk_count: int
    total_hexes: int
    oldest_access: int
    newest_access: int
    total_size_mb: float




#| export
@dataclass
class WatershedMeta:
    """Metadata for a stored watershed."""
    id: int = None
    world_id: int = 0
    name: str = ""
    # Terminal hex (outlet)
    terminal_q: int = 0
    terminal_r: int = 0
    terminal_s: int = 0
    is_ocean: bool = True
    # Stats
    total_flow: float = 0.0
    area_hexes: int = 0
    # Encoded data
    river_tree: str = ""   # River.encode()
    style: str = ""        # StyleCSS.encode()
    created: int = 0

    # NEW: Chunk identification (NULL for coarse-level watershed)
    chunk_q: Optional[int] = None
    chunk_r: Optional[int] = None
    chunk_s: Optional[int] = None

    modified: int = 0

In [ ]:
#| export


@dataclass
class World:
    """Metadata for a stored ChunkCober."""
    id: int = None
    name: str = ""
    extras: str = ""
    created: int = 0
    cover_data: str = ""  # Terrain.encode() for coarse map
    modified: int = 0




In [ ]:
#| export
@dataclass
class User:
    username: str
    email: str
    password: str
    created: int  # Unix timestamp
    sessionID: str
    activeWorld: int # a link into world_id for world 
    id: int = None

In [ ]:
#| export
@dataclass
class ChunkBorder:
    """Track drainage across chunk boundaries."""
    id: int = None
    world_id: int = 0
    chunk_q: int = 0
    chunk_r: int = 0  
    chunk_s: int = 0
    border_hex_q: int = 0  # Local hex at border
    border_hex_r: int = 0
    border_hex_s: int = 0
    downstream_chunk_q: int = 0  # Which chunk it drains to
    downstream_chunk_r: int = 0
    downstream_chunk_s: int = 0
    flow_volume: float = 0.0  # Accumulated upstream area


In [ ]:
#| export
@dataclass
class ChunkMeta:
    """Metadata for a cached chunk — avoids dimension inference."""
    id: int = None
    world_id: int = 0
    chunk_q: int = 0
    chunk_r: int = 0
    chunk_s: int = 0
    scale_level: int = 0
    nRows: int = 0
    nCols: int = 0
    radius: float = 0.0
    hex_count: int = 0        # How many valid hexes stored
    created: int = 0
    last_accessed: int = 0
    access_count: int = 0


## Main Storage

In [ ]:
#| export
class GeoStorage:

    def __init__(self, custom_path=None):
        path = GeoStorage.get_db_path(custom_path)
        self.path = path
        self.createDB()

   
    def createDB(self):
        db = database(self.path)
        self.db = db

        # Existing tables...
        db.create(World, pk='id', if_not_exists=True, transform=True)
        db.create(HexData, pk='id', if_not_exists=True, transform=True)
        db.create(User, pk='id', if_not_exists=True, transform=True)
        db.create(HexWeather, pk='id', if_not_exists=True, transform=True)
        db.create(ChunkBorder, pk='id', if_not_exists=True, transform=True)
        
        # NEW: Chunk metadata table
        db.create(ChunkMeta, pk='id', if_not_exists=True, transform=True)
        
        # Existing indices...
        db.execute("CREATE INDEX IF NOT EXISTS idx_hex_world ON hex_data(world_id)")
        db.execute("CREATE INDEX IF NOT EXISTS idx_hex_coords ON hex_data(world_id, q, r, s)")
        db.execute("CREATE INDEX IF NOT EXISTS idx_hex_grid ON hex_data(world_id, grid_index)")
        db.execute("CREATE INDEX IF NOT EXISTS idx_hex_temporal ON hex_data(world_id, q, r, s, modified DESC)")
        db.execute("CREATE INDEX IF NOT EXISTS idx_border_chunk ON chunk_border(world_id, chunk_q, chunk_r, chunk_s)")
        db.execute("CREATE INDEX IF NOT EXISTS idx_border_downstream ON chunk_border(world_id, downstream_chunk_q, downstream_chunk_r, downstream_chunk_s)")
        db.execute("CREATE INDEX IF NOT EXISTS idx_hex_chunk ON hex_data(world_id, chunk_q, chunk_r, chunk_s, scale_level)")
        db.execute("CREATE INDEX IF NOT EXISTS idx_weather_world ON hex_weather(world_id)")
        db.execute("CREATE INDEX IF NOT EXISTS idx_weather_coords ON hex_weather(world_id, q, r, s)")
        db.execute("CREATE INDEX IF NOT EXISTS idx_weather_temporal ON hex_weather(world_id, q, r, s, modified DESC)")
        db.execute("CREATE INDEX IF NOT EXISTS idx_weather_season ON hex_weather(world_id, season)")
        db.execute("CREATE INDEX IF NOT EXISTS idx_weather_chunk ON hex_weather(world_id, chunk_q, chunk_r, chunk_s)")
        db.execute("CREATE INDEX IF NOT EXISTS idx_weather_scale ON hex_weather(world_id, scale_level)")
        
        # NEW: Chunk meta index — unique lookup
        db.execute("""CREATE UNIQUE INDEX IF NOT EXISTS idx_chunk_meta_lookup 
                    ON chunk_meta(world_id, chunk_q, chunk_r, chunk_s, scale_level)""")

        self.weather = db.t.hex_weather
        self.users = db.t.user
        self.hexes = db.t.hex_data
        self.worlds = db.t.world
        self.borders = db.t.chunk_border
        self.chunk_meta = db.t.chunk_meta  # NEW



    @staticmethod
    def get_db_path(custom_path=None):
        """Get database path. Tries package data, falls back to user directory."""
        if custom_path:
            return custom_path
        
        # Try package data directory first
        try:
            db_dir = resources.files('HexMagic').joinpath('data/db')
            db_path = Path(db_dir) / 'hexmagic.db'
            # Test if writable
            db_path.parent.mkdir(parents=True, exist_ok=True)
            db_path.touch(exist_ok=True)
            return str(db_path)
        except (PermissionError, OSError):
            # Fallback to user directory
            data_dir = Path.home() / '.hexmagic' / 'data'
            data_dir.mkdir(parents=True, exist_ok=True)
            return str(data_dir / 'hexmagic.db')

    def __enter__(self):
        return self

    def __exit__(self, *args):
        self.db.close()

    
    @staticmethod
    def _get(obj, key):
        """Get attribute from dict or object."""
        return obj[key] if isinstance(obj, dict) else getattr(obj, key)

In [ ]:
#| export


#this might be something we don't export
@patch
def reset_database(self:GeoStorage):
    """Delete and recreate database. USE WITH CAUTION!"""
    db_path = Path(self.path)  # Convert string to Path
    
    if db_path.exists():
        db_path.unlink()

    self.createDB()



In [ ]:
#| export

@patch
def _latest_hex_subquery(self: GeoStorage, world_id: int, as_of: int = None) -> str:
    """SQL subquery for latest hex state, optionally at a point in time."""
    time_clause = f"AND modified <= {as_of}" if as_of else ""
    return f"""
        SELECT world_id, q, r, s, MAX(modified) as max_mod
        FROM hex_data
        WHERE world_id = {world_id} {time_clause}
        GROUP BY world_id, q, r, s
    """




In [ ]:
#| export
@patch
def query_hexes_latest(self: GeoStorage, world_id: int, as_of: int = None) -> list:
    """Get most recent version of each hex, optionally at a specific time."""
    subq = self._latest_hex_subquery(world_id, as_of)
    
    cursor = self.db.execute(f"""
        SELECT h.* FROM hex_data h
        INNER JOIN ({subq}) latest 
            ON h.world_id = latest.world_id 
            AND h.q = latest.q AND h.r = latest.r AND h.s = latest.s
            AND h.modified = latest.max_mod
    """)
    
    # Get column names from cursor description
    cols = [d[0] for d in cursor.description]
    rows = [dict(zip(cols, row)) for row in cursor.fetchall()]
    
    return rows


In [ ]:
#| export
@patch
def update_hex(self: GeoStorage, world_id: int, q: int, r: int, s: int,
               elevation: float = None, latitude: float = None, longitude: float = None,
               distance_from_coast: float = None, grid_index: int = None) -> SaveResult:
    """Append a new state for a hex (temporal update)."""
    now = int(datetime.now().timestamp())
    
    record = {
        'world_id': world_id,
        'q': q, 'r': r, 's': s,
        'modified': now
    }
    
    # Get previous state to preserve unchanged fields
    prev = self.query_hexes_in_radius(world_id, q, r, s, radius=0)
    if prev:
        prev = prev[0]
        record['grid_index'] = grid_index if grid_index is not None else GeoStorage._get(prev, 'grid_index')
        record['elevation'] = elevation if elevation is not None else GeoStorage._get(prev, 'elevation')
        record['latitude'] = latitude if latitude is not None else GeoStorage._get(prev, 'latitude')
        record['longitude'] = longitude if longitude is not None else GeoStorage._get(prev, 'longitude')
        record['distance_from_coast'] = distance_from_coast if distance_from_coast is not None else GeoStorage._get(prev, 'distance_from_coast')
    else:
        record['grid_index'] = grid_index or 0
        record['elevation'] = elevation or 0.0
        record['latitude'] = latitude
        record['longitude'] = longitude
        record['distance_from_coast'] = distance_from_coast
    
    try:
        self.hexes.insert(record)
        return SaveResult(None, 'updated', f'hex ({q},{r},{s}) at {now}')
    except Exception as e:
        return SaveResult(None, 'error', str(e))


In [ ]:
#| export
@patch
def query_hexes_in_radius(self: GeoStorage, world_id: int,
                          center_q: int, center_r: int, center_s: int,
                          radius: int, as_of: int = None) -> list:
    """Query latest hexes within radius of a center position."""
    subq = self._latest_hex_subquery(world_id, as_of)
    
    rows = list(self.db.execute(f"""
        SELECT h.* FROM hex_data h
        INNER JOIN ({subq}) latest 
            ON h.world_id = latest.world_id 
            AND h.q = latest.q AND h.r = latest.r AND h.s = latest.s
            AND h.modified = latest.max_mod
        WHERE h.q BETWEEN ? AND ? AND h.r BETWEEN ? AND ?
    """, [
        center_q - radius, center_q + radius,
        center_r - radius, center_r + radius
    ]).fetchall())
    
    if not rows:
        return []
    
    # Hardcode column order to avoid cursor.description issue
    cols = ['id', 'world_id', 'q', 'r', 's', 'grid_index', 'elevation', 
        'plate_id', 'latitude', 'longitude', 'distance_from_coast', 
        'watershed_id', 'modified']
    
    rows = [dict(zip(cols, row)) for row in rows]
    
    # Filter to actual hex distance
    return [row for row in rows
            if max(abs(row['q'] - center_q), abs(row['r'] - center_r), abs(row['s'] - center_s)) <= radius]


### World

In [ ]:
#| export
@patch
def save_cover(self: GeoStorage, cover: ChunkCover, name: str = "") -> SaveResult:
    """Save ChunkCover to database.
    
    Args:
        cover: ChunkCover to save
        name: Optional name for the world
    
    Returns:
        SaveResult(world_id, status, context)
    """
    now = int(datetime.now().timestamp())
    
    try:
        with self.db.conn:
            # Handle ID assignment
            if cover.ident < 0:
                # Get next available ID
                cursor = self.db.execute("SELECT MAX(id) FROM world")
                max_id = cursor.fetchone()[0]
                cover.ident = (max_id or 0) + 1
            
            # Check if exists
            existing = self.db.execute(
                "SELECT id FROM world WHERE id = ?", 
                [cover.ident]
            ).fetchone()
            
            encoded = cover.encode()
            
            record = {
                'id': cover.ident,
                'name': name or f"Cover_{cover.ident}",
                'cover_data': encoded,
                'modified': now
            }
            
            if existing:
                # Update existing
                record['created'] = self.worlds[cover.ident]['created']
                self.worlds.update(record)
                status_msg = 'updated'
            else:
                # Insert new
                record['created'] = now
                self.worlds.insert(record)
                status_msg = 'saved'
            
            return SaveResult(
                cover.ident, 
                status_msg, 
                f"ChunkCover {cover.rings}r+{cover.halo_rings}h, {len(cover.terrain.hexGrid.hexes)} hexes"
            )
    
    except Exception as e:
        return SaveResult(None, 'error', str(e))


@patch
def load_cover(self: GeoStorage, cover_id: int) -> LoadResult:
    """Load ChunkCover from database.
    
    Args:
        cover_id: ID of the cover to load
    
    Returns:
        LoadResult(ChunkCover, status, context)
    """
    try:
        world = self.worlds[cover_id]
        if not world:
            return LoadResult(None, 'not_found', f'Cover {cover_id} not found')
        
        cover_data = GeoStorage._get(world, 'cover_data')
        if not cover_data:
            return LoadResult(None, 'error', 'No cover_data in world record')
        
        cover = ChunkCover.decode(cover_data)
        cover.ident = cover_id
        cover.db = self  # Attach database reference
        
        name = GeoStorage._get(world, 'name')
        return LoadResult(
            cover, 
            'loaded', 
            f"{name}: {cover.rings}r+{cover.halo_rings}h, {len(cover.terrain.hexGrid.hexes)} hexes"
        )
    
    except Exception as e:
        return LoadResult(None, 'error', str(e))


In [ ]:
#| export
@patch
def save(self: ChunkCover, name: str = "") -> SaveResult:
    """Save this ChunkCover to its attached database.
    
    Args:
        name: Optional name for the world
    
    Returns:
        SaveResult with updated ident
    """
    if self.db is None:
        return SaveResult(None, 'error', 'No database attached to ChunkCover')
    
    result = self.db.save_cover(self, name=name)
    
    # Update ident if it was assigned
    if result.status in ('saved', 'updated') and result.id is not None:
        self.ident = result.id
    
    return result


## Chunk

In [ ]:
#| export
@patch
def load_or_generate_chunk(self: GeoStorage, 
                           cover: ChunkCover,
                           origin: int = None,
                           scale: int = 2,
                           method: str = 'bilinear',
                           octaves: int = 2,
                           persistence: float = 0.3,
                           detail_strength: float = 0.2,
                           force_regenerate: bool = False) -> LoadResult:
    """Load cached chunk or generate new one.
    
    Returns LoadResult(terrain, status, context) where:
    - status='cached' if loaded from DB
    - status='generated' if computed fresh
    """
    # 1. Compute chunk identifier
    grid = cover.terrain.hexGrid
    if origin is None:
        origin = grid.middle
    
    origin_pos = grid.index_to_hexposition(origin)
    chunk_key = f"{cover.ident}_{origin_pos.q}_{origin_pos.r}_{origin_pos.s}_{scale}"
    
    # 2. Check cache
    if not force_regenerate:
        cached = self._load_cached_chunk(cover.ident, origin_pos, scale)
        if cached.status == 'loaded':
            return LoadResult(cached.data, 'cached', cached.context)
    
    # 3. Generate chunk
    zoomed = cover.zoomChunkCombined(
        origin=origin,
        scale=scale,
        method=method,
        octaves=octaves,
        persistence=persistence,
        detail_strength=detail_strength
    )
    
    # 4. Save to cache
    save_result = self._save_chunk_cache(
        cover.ident, 
        origin_pos, 
        scale, 
        zoomed
    )
    # Periodic eviction check (every N generations)
    if random.random() < 0.1:  # 10% of the time
        self.evict_lru_chunks(cover.ident, max_chunks=100)
    
    return LoadResult(
        zoomed, 
        'generated', 
        f"{save_result.context}, scale={scale}"
    )


In [ ]:
#| export
@patch
def invalidate_chunk_cache(self: GeoStorage, cover_id: int) -> SaveResult:
    """Delete all cached chunks and their metadata for a cover."""
    try:
        with self.db.conn:
            cursor = self.db.execute("""
                DELETE FROM hex_data 
                WHERE world_id = ? AND scale_level > 0
            """, [cover_id])
            hex_count = cursor.rowcount
            
            cursor = self.db.execute("""
                DELETE FROM chunk_meta
                WHERE world_id = ?
            """, [cover_id])
            meta_count = cursor.rowcount
        
        return SaveResult(hex_count, 'invalidated', 
                         f'{hex_count} cached hexes, {meta_count} metadata records deleted')
    
    except Exception as e:
        return SaveResult(None, 'error', str(e))


In [ ]:
#| export
@patch
def _save_chunk_cache(self: GeoStorage,
                      cover_id: int,
                      origin_pos: HexPosition,
                      scale: int,
                      terrain: Terrain) -> SaveResult:
    """Save zoomed terrain as cached chunk, including metadata."""
    now = int(datetime.now().timestamp())
    grid = terrain.hexGrid
    
    try:
        with self.db.conn:
            count = 0
            for idx in range(len(terrain.elevations)):
                if idx in grid.invalidRegion:
                    continue
                
                pos = grid.index_to_hexposition(idx)
                
                record = {
                    'world_id': cover_id,
                    'q': pos.q,
                    'r': pos.r,
                    's': pos.s,
                    'chunk_q': origin_pos.q,
                    'chunk_r': origin_pos.r,
                    'chunk_s': origin_pos.s,
                    'grid_index': idx,
                    'elevation': float(terrain.elevations[idx]),
                    'scale_level': scale,
                    'modified': now
                }
                self.hexes.insert(record)
                count += 1
            
            # NEW: Save chunk metadata
            self.chunk_meta.insert({
                'world_id': cover_id,
                'chunk_q': origin_pos.q,
                'chunk_r': origin_pos.r,
                'chunk_s': origin_pos.s,
                'scale_level': scale,
                'nRows': grid.nRows,
                'nCols': grid.nCols,
                'radius': grid.radius,
                'hex_count': count,
                'created': now,
                'last_accessed': now,
                'access_count': 0
            })
            
            return SaveResult(count, 'saved', f'{count} hexes cached')
    
    except Exception as e:
        return SaveResult(None, 'error', str(e))


In [ ]:
#| export
@patch
def _load_cached_chunk(self: GeoStorage, 
                       cover_id: int,
                       origin_pos: HexPosition,
                       scale: int) -> LoadResult:
    """Load cached chunk hexes from database."""
    try:
        # Query hexes with matching chunk coordinates
        cursor = self.db.execute("""
            SELECT h.* FROM hex_data h
            INNER JOIN (
                SELECT world_id, q, r, s, MAX(modified) as max_mod
                FROM hex_data
                WHERE world_id = ? 
                  AND chunk_q = ? AND chunk_r = ? AND chunk_s = ?
                  AND scale_level = ?
                GROUP BY world_id, q, r, s
            ) latest 
                ON h.world_id = latest.world_id 
                AND h.q = latest.q AND h.r = latest.r AND h.s = latest.s
                AND h.modified = latest.max_mod
        """, [cover_id, origin_pos.q, origin_pos.r, origin_pos.s, scale])
        
        cols = [d[0] for d in cursor.description]
        rows = [dict(zip(cols, row)) for row in cursor.fetchall()]
        
        if not rows:
            return LoadResult(None, 'not_found', 'No cached chunk')

        if rows:  # Cache hit - update access stats
            now = int(datetime.now().timestamp())
            self.db.execute("""
                UPDATE hex_data 
                SET last_accessed = ?, access_count = access_count + 1
                WHERE world_id = ? AND chunk_q = ? AND chunk_r = ? AND chunk_s = ?
                AND scale_level = ?
            """, [now, cover_id, origin_pos.q, origin_pos.r, origin_pos.s, scale])
        
        # Reconstruct terrain from rows
        # (You'll need to infer grid dimensions from the data)
        terrain = self._reconstruct_terrain_from_rows(rows, cover_id)
        
        return LoadResult(terrain, 'loaded', f'{len(rows)} cached hexes')
    
    except Exception as e:
        return LoadResult(None, 'error', str(e))

In [ ]:
#| export
@patch
def evict_lru_chunks(self: GeoStorage, cover_id: int, 
                     max_chunks: int = 100) -> SaveResult:
    """Evict oldest chunks until under limit."""
    
    # Count current chunks
    count = self.db.execute("""
        SELECT COUNT(DISTINCT chunk_q || ',' || chunk_r || ',' || chunk_s) 
        FROM hex_data WHERE world_id = ? AND scale_level > 0
    """, [cover_id]).fetchone()[0]
    
    if count <= max_chunks:
        return SaveResult(0, 'ok', 'under limit')
    
    to_evict = count - max_chunks
    
    # Find oldest chunks
    oldest = self.db.execute("""
        SELECT DISTINCT chunk_q, chunk_r, chunk_s, MIN(last_accessed) as oldest
        FROM hex_data 
        WHERE world_id = ? AND scale_level > 0
        GROUP BY chunk_q, chunk_r, chunk_s
        ORDER BY oldest ASC
        LIMIT ?
    """, [cover_id, to_evict]).fetchall()
    
    # Delete them
    deleted = 0
    with self.db.conn:
        for row in oldest:
            self.db.execute("""
                DELETE FROM hex_data 
                WHERE world_id = ? AND chunk_q = ? AND chunk_r = ? AND chunk_s = ?
                  AND scale_level > 0
            """, [cover_id, row['chunk_q'], row['chunk_r'], row['chunk_s']])
            deleted += 1
    
    return SaveResult(deleted, 'evicted', f'{deleted} chunks removed')


### Region

In [ ]:
#| export
@patch
def query_hexes_in_radius(self: GeoStorage, world_id: int,
                          center_q: int, center_r: int, center_s: int,
                          radius: int, as_of: int = None) -> list:
    """Query latest hexes within radius of a center position."""
    subq = self._latest_hex_subquery(world_id, as_of)
    
    cursor = self.db.execute(f"""
        SELECT h.* FROM hex_data h
        INNER JOIN ({subq}) latest 
            ON h.world_id = latest.world_id 
            AND h.q = latest.q AND h.r = latest.r AND h.s = latest.s
            AND h.modified = latest.max_mod
        WHERE h.q BETWEEN ? AND ? AND h.r BETWEEN ? AND ?
    """, [
        center_q - radius, center_q + radius,
        center_r - radius, center_r + radius
    ])
    
    # Convert to dicts like query_hexes_latest does
    cols = [d[0] for d in cursor.description]
    rows = [dict(zip(cols, row)) for row in cursor.fetchall()]
    
    # Filter to actual hex distance
    result = []
    for row in rows:
        dq = abs(GeoStorage._get(row, 'q') - center_q)
        dr = abs(GeoStorage._get(row, 'r') - center_r)
        ds = abs(GeoStorage._get(row, 's') - center_s)
        dist = max(dq, dr, ds)
        if dist <= radius:
            result.append(row)
    
    return result


### Weather

In [ ]:
#| export
@patch
def save_zoomed_weather(self: GeoStorage, 
                        cover_id: int,
                        origin_pos: HexPosition,
                        scale: int,
                        terrain: Terrain,
                        climate_name: str = "",
                        season: str = "annual") -> SaveResult:
    """Save weather from zoomed terrain, tagged by chunk origin."""
    now = int(datetime.now().timestamp())
    grid = terrain.hexGrid
    invalid = getattr(grid, 'invalidRegion', set())
    
    if 'temperature' not in terrain.fields or 'precipitation' not in terrain.fields:
        return SaveResult(None, 'error', 'Missing weather fields')
    
    try:
        with self.db.conn:
            count = 0
            for idx in range(len(terrain.elevations)):
                if idx in invalid:
                    continue
                
                pos = grid.index_to_hexposition(idx)
                
                record = {
                    'world_id': cover_id,
                    'q': pos.q,
                    'r': pos.r,
                    's': pos.s,
                    'chunk_q': origin_pos.q,
                    'chunk_r': origin_pos.r,
                    'chunk_s': origin_pos.s,
                    'temperature': float(terrain.fields['temperature'][idx]),
                    'precipitation': float(terrain.fields['precipitation'][idx]),
                    'climate_name': climate_name,
                    'scale_level': scale,
                    'season': season,
                    'modified': now
                }
                
                # Add optional fields
                for field in ['humidity', 'climate_pet', 'aridity_index']:
                    if field in terrain.fields:
                        record[field] = float(terrain.fields[field][idx])
                
                self.weather.insert(record)
                count += 1
            
            return SaveResult(count, 'saved', f'{count} weather records for chunk ({origin_pos.q},{origin_pos.r},{origin_pos.s})')
    
    except Exception as e:
        return SaveResult(None, 'error', str(e))


@patch
def load_zoomed_weather(self: GeoStorage,
                        cover_id: int,
                        origin_pos: HexPosition,
                        scale: int,
                        terrain: Terrain,
                        season: str = "annual") -> LoadResult:
    """Load cached weather into zoomed terrain."""
    try:
        cursor = self.db.execute("""
            SELECT hw.* FROM hex_weather hw
            INNER JOIN (
                SELECT world_id, q, r, s, MAX(modified) as max_mod
                FROM hex_weather
                WHERE world_id = ? AND season = ?
                  AND chunk_q = ? AND chunk_r = ? AND chunk_s = ?
                  AND scale_level = ?
                GROUP BY world_id, q, r, s
            ) latest 
                ON hw.world_id = latest.world_id 
                AND hw.q = latest.q AND hw.r = latest.r AND hw.s = latest.s
                AND hw.modified = latest.max_mod
        """, [cover_id, season, origin_pos.q, origin_pos.r, origin_pos.s, scale])
        
        cols = [d[0] for d in cursor.description]
        rows = [dict(zip(cols, row)) for row in cursor.fetchall()]
        
        if not rows:
            return LoadResult(terrain, 'not_found', 'No cached weather')
        
        # Initialize fields
        n = len(terrain.elevations)
        for field in ['temperature', 'precipitation', 'humidity', 'climate_pet', 'aridity_index']:
            if field not in terrain.fields:
                terrain.fields[field] = np.zeros(n)
        
        grid = terrain.hexGrid
        filled = 0
        
        for row in rows:
            pos = HexPosition(row['q'], row['r'], row['s'])
            idx = grid.hexposition_to_index(pos)
            
            if 0 <= idx < n:
                terrain.fields['temperature'][idx] = row['temperature'] or 0.0
                terrain.fields['precipitation'][idx] = row['precipitation'] or 0.0
                if row.get('humidity') is not None:
                    terrain.fields['humidity'][idx] = row['humidity']
                if row.get('climate_pet') is not None:
                    terrain.fields['climate_pet'][idx] = row['climate_pet']
                if row.get('aridity_index') is not None:
                    terrain.fields['aridity_index'][idx] = row['aridity_index']
                filled += 1
        
        return LoadResult(terrain, 'loaded', f'{filled} weather records')
    
    except Exception as e:
        return LoadResult(terrain, 'error', str(e))


@patch
def has_zoomed_weather(self: GeoStorage,
                       cover_id: int,
                       origin_pos: HexPosition,
                       scale: int,
                       season: str = "annual",
                       min_count: int = 1) -> bool:
    """Check if cached weather exists for this zoom."""
    try:
        cursor = self.db.execute("""
            SELECT COUNT(*) FROM hex_weather
            WHERE world_id = ? AND season = ?
              AND chunk_q = ? AND chunk_r = ? AND chunk_s = ?
              AND scale_level = ?
        """, [cover_id, season, origin_pos.q, origin_pos.r, origin_pos.s, scale])
        
        count = cursor.fetchone()[0]
        return count >= min_count
    except Exception:
        return False


### watershed

### Chunk Cover

#| export
@patch
def _reconstruct_terrain_from_rows(self: GeoStorage, rows: list[dict], cover_id: int) -> Terrain:
    """Reconstruct Terrain from cached hex_data rows."""
    if not rows:
        return None
    
    # Infer grid dimensions from row data
    qs = [r['q'] for r in rows]
    rs = [r['r'] for r in rows]
    
    # Get original cover to retrieve grid params
    world = self.worlds[cover_id]
    cover_data = GeoStorage._get(world, 'cover_data')
    if cover_data:
        # Extract just the grid info from cover
        original_cover = ChunkCover.decode(cover_data)
        base_radius = original_cover.terrain.hexGrid.radius
    else:
        base_radius = 10  # fallback
    
    # Compute grid size from coordinate ranges
    nRows = max(rs) - min(rs) + 1
    nCols = max(qs) - min(qs) + 1
    
    # Scale factor stored in rows
    scale = rows[0].get('scale_level', 1)
    radius = base_radius / scale
    
    grid = HexGrid(nRows=nRows, nCols=nCols, radius=radius, style=StyleCSS("simple"))
    
    terrain = Terrain.__new__(Terrain)
    terrain.hexGrid = grid
    terrain.elevations = np.zeros(len(grid.hexes))
    terrain.fields = {}
    
    # Map rows to terrain
    for row in rows:
        pos = HexPosition(row['q'], row['r'], row['s'])
        idx = grid.hexposition_to_index(pos)
        if 0 <= idx < len(terrain.elevations):
            terrain.elevations[idx] = row['elevation']
    
    return terrain


In [ ]:
#| export
@patch
def zoom_cached(self: ChunkCover, 
                origin: int = None,
                scale: int = 2,
                **kwargs) -> Terrain:
    """Zoom with database caching (requires db attached)."""
    if self.db is None:
        # Fallback to direct generation
        return self.zoomChunkCombined(origin=origin, scale=scale, **kwargs)
    
    result = self.db.load_or_generate_chunk(
        self, 
        origin=origin, 
        scale=scale, 
        **kwargs
    )
    
    if result.status == 'cached':
        print(f"✓ Loaded from cache: {result.context}")
    else:
        print(f"⚙ Generated: {result.context}")
    
    return result.data


In [ ]:
#| export
@patch
def zoom_with_full_data(self: ChunkCover,
                        origin: int = None,
                        scale: int = 2,
                        compute_weather: bool = True,
                        force_regenerate: bool = False) -> DrainageBasins:
    """Zoom chunk with weather and watersheds, using cache where possible.
    
    Returns DrainageBasins with terrain set to the zoomed terrain.
    """
    if self.db is None:
        raise ValueError("ChunkCover.db must be set for caching")
    
    grid = self.terrain.hexGrid
    if origin is None:
        origin = grid.middle
    origin_pos = grid.index_to_hexposition(origin)
    
    # Step 1: Lazy init coarse basin (O(n²) only once)
    if self.basin is None:
        self.basin = DrainageBasins(self.terrain)
        # Save cover with basin to persist
        self.save()
    
    # Step 2: Base terrain (cached)
    result = self.db.load_or_generate_chunk(
        self, origin=origin, scale=scale, force_regenerate=force_regenerate
    )
    zoomed = result.data
    
    # Step 3: Weather
    if compute_weather:
        if not force_regenerate and self.db.has_zoomed_weather(self.ident, origin_pos, scale):
            self.db.load_zoomed_weather(self.ident, origin_pos, scale, zoomed)
        else:
            zoomed.climate = self.terrain.climate
            zoomed.geo = self.terrain.geo
            zoomed.compute_weather()
            self.db.save_zoomed_weather(self.ident, origin_pos, scale, zoomed)
    
    # Step 4: Build fine_regions mapping (O(n), cheap)
    fine_regions = self._build_fine_regions_map(origin, scale, zoomed.hexGrid)
    
    # Step 5: Project watersheds (O(n), uses cached coarse basin)
    zoomed_sheds = self.project_watersheds(self.basin, fine_regions, zoomed)
    
    # Step 6: Return DrainageBasins with zoomed terrain
    zoomed_basins = DrainageBasins(zoomed, compute=False)
    zoomed_basins.sheds = zoomed_sheds
    
    return zoomed_basins


@patch
def _build_fine_regions_map(self: ChunkCover, 
                            origin: int, 
                            scale: int, 
                            fine_grid: HexGrid) -> dict[int, set[int]]:
    """Map coarse hex indices to sets of fine hex indices.
    
    O(n) — just coordinate math, no flow computation.
    """
    coarse_grid = self.terrain.hexGrid
    extent = 3 * self.rings + self.halo_rings
    
    center_row = origin // coarse_grid.nCols
    center_col = origin % coarse_grid.nCols
    
    fine_regions = {}
    invalid = getattr(fine_grid, 'invalidRegion', set())
    
    for coarse_row in range(center_row - extent, center_row + extent + 1):
        for coarse_col in range(center_col - extent, center_col + extent + 1):
            if not (0 <= coarse_row < coarse_grid.nRows and 0 <= coarse_col < coarse_grid.nCols):
                continue
            
            coarse_idx = coarse_row * coarse_grid.nCols + coarse_col
            
            fine_start_row = (coarse_row - (center_row - extent)) * scale
            fine_start_col = (coarse_col - (center_col - extent)) * scale
            
            fine_hexes = set()
            for dr in range(scale):
                for dc in range(scale):
                    fine_row = fine_start_row + dr
                    fine_col = fine_start_col + dc
                    if 0 <= fine_row < fine_grid.nRows and 0 <= fine_col < fine_grid.nCols:
                        fine_idx = fine_row * fine_grid.nCols + fine_col
                        if fine_idx not in invalid:
                            fine_hexes.add(fine_idx)
            
            if fine_hexes:
                fine_regions[coarse_idx] = fine_hexes
    
    return fine_regions


In [ ]:
#| export
@patch
def invalidate_all_caches(self: GeoStorage, cover_id: int) -> dict:
    """Invalidate all cached data when coarse terrain changes."""
    results = {}

     # Use transaction for atomic invalidation
    try:
        with self.db.conn:
            
            # Invalidate terrain chunks
            results['terrain'] = self.invalidate_chunk_cache(cover_id)
            
            # Invalidate weather
            cursor = self.db.execute("""
                DELETE FROM hex_weather 
                WHERE world_id = ? AND scale_level > 0
            """, [cover_id])
            results['weather'] = SaveResult(cursor.rowcount, 'invalidated', f'{cursor.rowcount} weather records')
            
            # Invalidate watersheds
            cursor = self.db.execute("""
                DELETE FROM hex_data 
                WHERE world_id = ? AND scale_level > 0 AND watershed_id IS NOT NULL
            """, [cover_id])
            results['watersheds'] = SaveResult(cursor.rowcount, 'invalidated', f'{cursor.rowcount} watershed assignments')
        
            return results
    except Exception as e:
        # Transaction rolled back automatically
        return {'error': SaveResult(None, 'error', str(e))}


In [ ]:
#| export
# === WATERSHED PERSISTENCE ===

@patch
def save_chunk_watersheds(self: GeoStorage, chunk: HexChunk, 
                          chunk_ref: ChunkRef, world_id: int) -> SaveResult:
    """Save watershed assignments for a chunk."""
    now = int(datetime.now().timestamp())
    p = chunk_ref.position
    
    if 'watershed_id' not in chunk.fields:
        return SaveResult(None, 'error', 'Chunk missing watershed_id field')
    
    try:
        with self.db.conn:
            count = 0
            for idx in chunk.iter_core():
                world_pos = chunk.index_to_world(idx)
                ws_id = int(chunk.fields['watershed_id'][idx])
                
                if ws_id < 0:
                    continue  # Skip unassigned/ocean
                
                # Update hex_data with watershed_id
                self.hexes.insert({
                    'world_id': world_id,
                    'q': world_pos.q,
                    'r': world_pos.r,
                    's': world_pos.s,
                    'grid_index': idx,
                    'elevation': float(chunk.elevations[idx]),
                    'watershed_id': ws_id,
                    'modified': now
                })
                count += 1
            
            return SaveResult(count, 'saved', 
                            f'{count} watershed assignments for chunk ({p.q},{p.r},{p.s})')
    except Exception as e:
        return SaveResult(None, 'error', str(e))


@patch
def load_chunk_watersheds(self: GeoStorage, world_id: int,
                          chunk: HexChunk, chunk_ref: ChunkRef) -> LoadResult:
    """Load watershed assignments for a chunk."""
    try:
        p = chunk_ref.position
        
        # Query hexes with watershed_id for this chunk's world positions
        hex_rows = []
        for idx in chunk.iter_core():
            world_pos = chunk.index_to_world(idx)
            rows = self.query_hexes_in_radius(world_id, world_pos.q, world_pos.r, world_pos.s, radius=0)
            if rows:
                hex_rows.append((idx, rows[0]))
        
        if not hex_rows:
            return LoadResult(chunk, 'not_found', 
                            f'No watershed data for chunk ({p.q},{p.r},{p.s})')
        
        # Initialize field if needed
        if 'watershed_id' not in chunk.fields:
            chunk.add_field('watershed_id', default=-1)
        
        filled = 0
        for idx, row in hex_rows:
            ws_id = row.get('watershed_id')
            if ws_id is not None and ws_id >= 0:
                chunk.fields['watershed_id'][idx] = ws_id
                filled += 1
        
        return LoadResult(chunk, 'loaded', 
                         f'{filled} watershed assignments for chunk ({p.q},{p.r},{p.s})')
    except Exception as e:
        return LoadResult(None, 'error', str(e))


In [ ]:
#| export
@patch
def has_chunk_watersheds(self: GeoStorage, world_id: int, 
                         chunk_ref: ChunkRef,
                         min_count: int = 1) -> bool:
    """Check if watershed data exists for chunk."""
    try:
        center = chunk_ref.center_hex
        # Use larger radius to cover more of the chunk
        radius = chunk_ref.rings  # Full chunk radius, not just 3
        
        rows = self.query_hexes_in_radius(world_id, center.q, center.r, center.s, radius=radius)
        
        ws_count = sum(1 for r in rows if r.get('watershed_id') is not None and r.get('watershed_id') >= 0)
        return ws_count >= min_count
    except Exception:
        return False



## Debugger

In [ ]:
#| export
class GeoStorageDebugger:
    """Test harness for HexServer with automatic cleanup."""
    
    def __init__(self, keep_on_error=False):
        """
        Args:
            keep_on_error: If True, preserve database when tests fail
        """
        self.temp_dir = tempfile.mkdtemp(prefix='hexmagic_test_')
        self.db_path = os.path.join(self.temp_dir, 'test.db')
        self.server = GeoStorage(custom_path=self.db_path)
        self.keep_on_error = keep_on_error
        self.failed = False
        
    def preserve_db(self, test_name):
        """Copy database to inspection directory."""
        inspect_dir = Path.home() / '.hexmagic' / 'test_failures'
        inspect_dir.mkdir(parents=True, exist_ok=True)
        
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        dest = inspect_dir / f"{test_name}_{timestamp}.db"
        
        shutil.copy2(self.db_path, dest)
        print(f"⚠ Database preserved at: {dest}")
        
    def close(self):
        """Clean up test database."""
        self.server.db.close()
        
        if not self.failed or not self.keep_on_error:
            shutil.rmtree(self.temp_dir)
            print("✓ Test database cleaned up")
        else:
            print(f"⚠ Test database kept at: {self.temp_dir}")
            
    def __enter__(self):
        return self
        
    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is not None:
            self.failed = True
        self.close()


## The Big refactor

In [ ]:
# Test save/load with GeoStorageDebugger
with GeoStorageDebugger(keep_on_error=True) as dbg:
    # 1. Create terrain and cover
    terrain = TerraDemo().maui_map()
    cover = ChunkCover(terrain, rings=3, halo_rings=1)
    cover.db = dbg.server  # Attach database
    
    print(f"Original: {len(terrain.hexGrid.hexes)} hexes, {terrain.hexGrid.nRows}x{terrain.hexGrid.nCols}")
    
    # 2. Save
    result = cover.save(name="Maui Test")
    print(f"Save: {result.status} - {result.context}")
    cover_id = result.id
    
    # 3. Load back
    loaded = dbg.server.load_cover(cover_id)
    print(f"Load: {loaded.status} - {loaded.context}")
    
    # 4. Verify
    orig_elev = cover.terrain.elevations
    load_elev = loaded.data.terrain.elevations
    
    match = np.allclose(orig_elev, load_elev)
    print(f"Elevations match: {match}")
    
    if match:
        print("✓ Round-trip test passed!")
    else:
        dbg.failed = True
        print("✗ Data mismatch!")


In [ ]:
with GeoStorageDebugger(keep_on_error=True) as dbg:
    terrain = TerraDemo().maui_map()
    cover = ChunkCover(terrain, rings=3, halo_rings=1)
    cover.db = dbg.server
    cover.save(name="Maui Chunk Test")
    
    # First zoom - should generate
    result1 = dbg.server.load_or_generate_chunk(cover, origin=None, scale=2)
    print(f"First call: {result1.status} - {result1.context}")
    
    # Second zoom - should hit cache
    result2 = dbg.server.load_or_generate_chunk(cover, origin=None, scale=2)
    print(f"Second call: {result2.status} - {result2.context}")
    
    if result1.status == 'generated' and result2.status == 'cached':
        print("✓ Chunk caching works!")
    else:
        print("✗ Caching issue")


In [ ]:
with GeoStorageDebugger(keep_on_error=True) as dbg:
    terrain = TerraDemo().maui_map()
    terrain.compute_weather()  # Need weather for full data
    
    cover = ChunkCover(terrain, rings=3, halo_rings=1)
    cover.db = dbg.server
    cover.save(name="Maui Full Test")
    
    print(f"Basin before: {cover.basin}")
    
    # zoom_with_full_data should:
    # 1. Lazy-init basin (O(n²) once)
    # 2. Generate zoomed terrain
    # 3. Project watersheds (O(n))
    basins = cover.zoom_with_full_data(origin=None, scale=2, compute_weather=True)
    
    print(f"Basin after: {cover.basin is not None}")
    print(f"Zoomed terrain: {basins.terrain.hexGrid.nRows}x{basins.terrain.hexGrid.nCols}")
    print(f"Watersheds: {len(basins.sheds)}")
    
    # Check weather was computed
    has_temp = 'temperature' in basins.terrain.fields
    print(f"Has temperature: {has_temp}")
    
    if cover.basin is not None and len(basins.sheds) > 0 and has_temp:
        print("✓ zoom_with_full_data works!")
    else:
        print("✗ Something missing")


In [ ]:
with GeoStorageDebugger(keep_on_error=True) as dbg:
    terrain = TerraDemo().maui_map()
    terrain.compute_weather()  # Need weather for full data
    
    cover = ChunkCover(terrain, rings=3, halo_rings=1)
    cover.db = dbg.server
    cover.save(name="Maui Full Test")
    
    print(f"Basin before: {cover.basin}")
    
    # zoom_with_full_data should:
    # 1. Lazy-init basin (O(n²) once)
    # 2. Generate zoomed terrain
    # 3. Project watersheds (O(n))
    basins = cover.zoom_with_full_data(origin=None, scale=2, compute_weather=True)
    
    print(f"Basin after: {cover.basin is not None}")
    print(f"Zoomed terrain: {basins.terrain.hexGrid.nRows}x{basins.terrain.hexGrid.nCols}")
    print(f"Watersheds: {len(basins.sheds)}")
    
    # Check weather was computed
    has_temp = 'temperature' in basins.terrain.fields
    print(f"Has temperature: {has_temp}")
terrain = basins.terrain
terrain.colorMap()
terrain.hexGrid.update()
#terrain.hexGrid.builder.show()

    
   


## By Region

In [ ]:
!cat ../HexMagic/plot/*.py

In [ ]:
!cat ../HexMagic/terrain.py

In [ ]:
??HexRegion.crop_to_centered_grid

I want to build something similar to crop_to_centered_grid but using the mainTable and the database. so instead of creating a subsection of the main we would return a detail terrain, drainagebasin,and mapping that would link the new terrain back to hexes on the original terrain. Does this make sense?

Lets input based upon a HexRegion. we would need to figure out which chunks. I think just a fine->coarse since the reverse will have many destinations


I think we want option A. 

So what I am thinking is pick the first hex in the region and find its region. then uses set difference operator to remove all of the hexes in the region that are in that chunk. continue until we have the chunks. then the question is how do we reassemble them. maybe we can take advanage of the cubecoordinates to figure out the average of the chunks which would be our orign. we can figure out the max east and west using the relavivet postions. north and south are harder and we might have to do some y coordinate math to get those.

I think b is better. The goal is to get to a terrain and have a drainage basin (using the chunkcover approach so we don't do o n^2 operations

So the mapAround shows how I stitch around a single chunk to get a terrain. Does the stiching make sense for a set of chunks

So I do think we should make this work on contiguous regions it is pussible this could be in a u shape for instance in which case we would need to fill in the missing hexes. does it make sense to get the bounding box of the hexRegion and then find the chunks?

option b. We want to save invalid for where the map can't reprsent

from dataclasses import dataclass
from typing import Callable

@dataclass
class ZoomResult:
    """Result of zooming into a region."""
    terrain: Terrain
    basins: DrainageBasins
    mapper: Callable[[int], int]  # fine_idx → coarse_idx (-1 if invalid)
    chunks_loaded: int


def region_bounding_box(region: HexRegion) -> tuple[int, int, int, int]:
    """Get (min_row, max_row, min_col, max_col) for a region."""
    grid = region.hexGrid
    rows = []
    cols = []
    for idx in region.hexes:
        row, col = grid.index_to_row_col(idx)
        rows.append(row)
        cols.append(col)
    return min(rows), max(rows), min(cols), max(cols)


def bbox_to_chunk_refs(min_row: int, max_row: int, min_col: int, max_col: int,
                       cover: ChunkCover) -> set[ChunkRef]:
    """Find all ChunkRefs that overlap a bounding box."""
    grid = cover.terrain.hexGrid
    chunk_rings = cover.rings
    spacing = chunk_rings * 2
    
    chunks = set()
    
    # Convert row/col range to indices, then to chunk positions
    for row in range(min_row, max_row + 1, spacing):
        for col in range(min_col, max_col + 1, spacing):
            idx = grid.row_col_to_index(row, col)
            if idx < 0:
                continue
            
            # Convert to HexPosition, then to chunk position
            world_pos = grid.index_to_hexposition(idx, origin_index=0)
            chunk_pos, _ = world_to_chunk(world_pos, chunk_rings)
            chunks.add(ChunkRef(chunk_pos, chunk_rings))
    
    # Also check corners and edges to ensure coverage
    for row in [min_row, max_row]:
        for col in [min_col, max_col]:
            idx = grid.row_col_to_index(row, col)
            if idx >= 0:
                world_pos = grid.index_to_hexposition(idx, origin_index=0)
                chunk_pos, _ = world_to_chunk(world_pos, chunk_rings)
                chunks.add(ChunkRef(chunk_pos, chunk_rings))
    
    return chunks


def stitch_chunks(chunks: set[ChunkRef], cover: ChunkCover, 
                  storage: GeoStorage, scale: int) -> tuple[Terrain, dict[tuple, tuple]]:
    """Load chunks and stitch into single terrain.
    
    Returns (merged_terrain, chunk_offsets) where chunk_offsets maps
    chunk.key → (row_offset, col_offset) in merged grid.
    """
    if not chunks:
        return None, {}
    
    # Load all chunk terrains
    chunk_terrains = {}
    for chunk in chunks:
        origin_idx = cover.terrain.hexGrid.hexposition_to_index(chunk.center_hex, origin_index=0)
        if origin_idx < 0:
            # Chunk center is outside grid, skip
            continue
        result = storage.load_or_generate_chunk(cover, origin=origin_idx, scale=scale)
        if result.data is not None:
            chunk_terrains[chunk.key] = result.data
    
    if not chunk_terrains:
        return None, {}
    
    # Get chunk grid dimensions (all should be same size)
    sample_terrain = next(iter(chunk_terrains.values()))
    chunk_rows = sample_terrain.hexGrid.nRows
    chunk_cols = sample_terrain.hexGrid.nCols
    chunk_radius = sample_terrain.hexGrid.radius
    
    # Find bounding box of chunks in chunk-space
    chunk_qs = [k[0] for k in chunk_terrains.keys()]
    chunk_rs = [k[1] for k in chunk_terrains.keys()]
    
    min_q, max_q = min(chunk_qs), max(chunk_qs)
    min_r, max_r = min(chunk_rs), max(chunk_rs)
    
    # Calculate merged grid size
    n_chunks_q = max_q - min_q + 1
    n_chunks_r = max_r - min_r + 1
    
    merged_rows = n_chunks_r * chunk_rows
    merged_cols = n_chunks_q * chunk_cols
    
    # Create merged terrain
    merged_grid = HexGrid(
        nRows=merged_rows, 
        nCols=merged_cols, 
        radius=chunk_radius, 
        style=sample_terrain.hexGrid.style
    )
    
    merged_terrain = Terrain(bounds=merged_grid.bounds, radius=chunk_radius)
    merged_terrain.hexGrid = merged_grid
    merged_terrain.elevations = np.full(merged_rows * merged_cols, -100.0)  # Default to water
    merged_terrain.fields = {}
    
    # Initialize fields from sample
    for field_name in sample_terrain.fields:
        merged_terrain.fields[field_name] = np.zeros(merged_rows * merged_cols)
    
    # Copy climate/geo if present
    if hasattr(sample_terrain, 'climate'):
        merged_terrain.climate = sample_terrain.climate
    if hasattr(sample_terrain, 'geo'):
        merged_terrain.geo = sample_terrain.geo
    
    # Track chunk offsets
    chunk_offsets = {}
    
    # Copy each chunk into merged terrain
    for key, chunk_terrain in chunk_terrains.items():
        q, r, s = key
        
        # Offset in merged grid
        row_offset = (r - min_r) * chunk_rows
        col_offset = (q - min_q) * chunk_cols
        chunk_offsets[key] = (row_offset, col_offset)
        
        chunk_grid = chunk_terrain.hexGrid
        
        for local_idx in range(len(chunk_terrain.elevations)):
            local_row, local_col = chunk_grid.index_to_row_col(local_idx)
            
            merged_row = row_offset + local_row
            merged_col = col_offset + local_col
            merged_idx = merged_row * merged_cols + merged_col
            
            if 0 <= merged_idx < len(merged_terrain.elevations):
                merged_terrain.elevations[merged_idx] = chunk_terrain.elevations[local_idx]
                for field_name, field_data in chunk_terrain.fields.items():
                    merged_terrain.fields[field_name][merged_idx] = field_data[local_idx]
    
    return merged_terrain, chunk_offsets


def build_fine_to_coarse_mapper(cover: ChunkCover, 
                                merged_terrain: Terrain,
                                chunk_offsets: dict,
                                scale: int) -> Callable[[int], int]:
    """Build function mapping fine terrain index → coarse terrain index."""
    coarse_grid = cover.terrain.hexGrid
    fine_grid = merged_terrain.hexGrid
    
    # Precompute for efficiency
    chunk_rows = fine_grid.nRows // len(set(k[1] for k in chunk_offsets.keys())) if chunk_offsets else fine_grid.nRows
    chunk_cols = fine_grid.nCols // len(set(k[0] for k in chunk_offsets.keys())) if chunk_offsets else fine_grid.nCols
    
    # Invert chunk_offsets for lookup
    offset_to_chunk = {v: k for k, v in chunk_offsets.items()}
    
    def mapper(fine_idx: int) -> int:
        fine_row, fine_col = fine_grid.index_to_row_col(fine_idx)
        
        # Which chunk is this in?
        chunk_row_idx = fine_row // chunk_rows
        chunk_col_idx = fine_col // chunk_cols
        
        # Find corresponding chunk key
        row_offset = chunk_row_idx * chunk_rows
        col_offset = chunk_col_idx * chunk_cols
        
        chunk_key = offset_to_chunk.get((row_offset, col_offset))
        if chunk_key is None:
            return -1
        
        q, r, s = chunk_key
        chunk_center = HexPosition(q, r, s) * (cover.rings * 2)
        
        # Local position within chunk
        local_row = fine_row - row_offset
        local_col = fine_col - col_offset
        
        # Scale down to coarse
        coarse_local_row = local_row // scale
        coarse_local_col = local_col // scale
        
        # Convert to coarse grid index
        # The chunk center in coarse grid
        center_idx = coarse_grid.hexposition_to_index(chunk_center, origin_index=0)
        if center_idx < 0:
            return -1
        
        center_row, center_col = coarse_grid.index_to_row_col(center_idx)
        
        # Offset from chunk center
        half_chunk = cover.rings
        coarse_row = center_row - half_chunk + coarse_local_row
        coarse_col = center_col - half_chunk + coarse_local_col
        
        return coarse_grid.row_col_to_index(coarse_row, coarse_col)
    
    return mapper


@patch
def zoom_region(self: ChunkCover, 
                region: HexRegion,
                scale: int = 2,
                compute_weather: bool = True) -> ZoomResult:
    """Zoom into a region with stitched chunks and projected watersheds.
    
    Args:
        region: HexRegion on the coarse terrain
        scale: Zoom factor
        compute_weather: Whether to compute weather on zoomed terrain
    
    Returns:
        ZoomResult with terrain, basins, and fine→coarse mapper
    """
    if self.db is None:
        raise ValueError("ChunkCover.db must be set for zoom_region")
    
    # Step 1: Get bounding box
    min_row, max_row, min_col, max_col = region_bounding_box(region)
    
    # Add padding for halo
    padding = self.halo_rings + 1
    min_row = max(0, min_row - padding)
    max_row = min(self.terrain.hexGrid.nRows - 1, max_row + padding)
    min_col = max(0, min_col - padding)
    max_col = min(self.terrain.hexGrid.nCols - 1, max_col + padding)
    
    # Step 2: Find chunks
    chunks = bbox_to_chunk_refs(min_row, max_row, min_col, max_col, self)
    
    # Step 3 & 4: Load and stitch
    merged_terrain, chunk_offsets = stitch_chunks(chunks, self, self.db, scale)
    
    if merged_terrain is None:
        return ZoomResult(None, None, lambda x: -1, 0)
    
    # Mark invalid regions (outside original terrain bounds)
    coarse_grid = self.terrain.hexGrid
    fine_grid = merged_terrain.hexGrid
    
    for fine_idx in range(len(merged_terrain.elevations)):
        fine_row, fine_col = fine_grid.index_to_row_col(fine_idx)
        coarse_row = fine_row // scale
        coarse_col = fine_col // scale
        
        # Check if corresponding coarse hex exists
        if coarse_row < 0 or coarse_row >= coarse_grid.nRows or \
           coarse_col < 0 or coarse_col >= coarse_grid.nCols:
            fine_grid.invalidRegion.add(fine_idx)
    
    # Step 5: Weather
    if compute_weather and hasattr(self.terrain, 'climate'):
        merged_terrain.climate = self.terrain.climate
        merged_terrain.geo = getattr(self.terrain, 'geo', None)
        merged_terrain.compute_weather()
    
    # Step 6: Lazy init coarse basin if needed
    if self.basin is None:
        self.basin = DrainageBasins(self.terrain)
        self.save()
    
    # Build fine_regions for watershed projection
    fine_regions = {}
    mapper = build_fine_to_coarse_mapper(self, merged_terrain, chunk_offsets, scale)
    
    for fine_idx in range(len(merged_terrain.elevations)):
        if fine_idx in fine_grid.invalidRegion:
            continue
        coarse_idx = mapper(fine_idx)
        if coarse_idx >= 0:
            if coarse_idx not in fine_regions:
                fine_regions[coarse_idx] = set()
            fine_regions[coarse_idx].add(fine_idx)
    
    # Step 7: Project watersheds
    zoomed_sheds = self.project_watersheds(self.basin, fine_regions, merged_terrain)
    
    # Build DrainageBasins
    zoomed_basins = DrainageBasins(merged_terrain, compute=False)
    zoomed_basins.sheds = zoomed_sheds
    
    return ZoomResult(
        terrain=merged_terrain,
        basins=zoomed_basins,
        mapper=mapper,
        chunks_loaded=len(chunk_offsets)
    )


In [ ]:
#| export
from dataclasses import dataclass
from typing import Callable

@dataclass
class ZoomResult:
    """Result of zooming into a region."""
    terrain: Terrain
    basins: DrainageBasins
    mapper: Callable[[int], int]  # fine_idx → coarse_idx (-1 if invalid)
    chunks_loaded: int


def region_bounding_box(region: HexRegion) -> tuple[int, int, int, int]:
    """Get (min_row, max_row, min_col, max_col) for a region."""
    grid = region.hexGrid
    rows = []
    cols = []
    for idx in region.hexes:
        row, col = grid.index_to_row_col(idx)
        rows.append(row)
        cols.append(col)
    return min(rows), max(rows), min(cols), max(cols)


def bbox_to_chunk_refs(min_row: int, max_row: int, min_col: int, max_col: int,
                       cover: ChunkCover) -> set[ChunkRef]:
    """Find all ChunkRefs that overlap a bounding box."""
    grid = cover.terrain.hexGrid
    chunk_rings = cover.rings
    spacing = chunk_rings * 2
    
    chunks = set()
    
    # Convert row/col range to indices, then to chunk positions
    for row in range(min_row, max_row + 1, spacing):
        for col in range(min_col, max_col + 1, spacing):
            idx = grid.row_col_to_index(row, col)
            if idx < 0:
                continue
            
            # Convert to HexPosition, then to chunk position
            world_pos = grid.index_to_hexposition(idx, origin_index=0)
            chunk_pos, _ = world_to_chunk(world_pos, chunk_rings)
            chunks.add(ChunkRef(chunk_pos, chunk_rings))
    
    # Also check corners and edges to ensure coverage
    for row in [min_row, max_row]:
        for col in [min_col, max_col]:
            idx = grid.row_col_to_index(row, col)
            if idx >= 0:
                world_pos = grid.index_to_hexposition(idx, origin_index=0)
                chunk_pos, _ = world_to_chunk(world_pos, chunk_rings)
                chunks.add(ChunkRef(chunk_pos, chunk_rings))
    
    return chunks



def build_fine_to_coarse_mapper(cover: ChunkCover, 
                                merged_terrain: Terrain,
                                chunk_offsets: dict,
                                scale: int) -> Callable[[int], int]:
    """Build function mapping fine terrain index → coarse terrain index."""
    coarse_grid = cover.terrain.hexGrid
    fine_grid = merged_terrain.hexGrid
    
    # Precompute for efficiency
    chunk_rows = fine_grid.nRows // len(set(k[1] for k in chunk_offsets.keys())) if chunk_offsets else fine_grid.nRows
    chunk_cols = fine_grid.nCols // len(set(k[0] for k in chunk_offsets.keys())) if chunk_offsets else fine_grid.nCols
    
    # Invert chunk_offsets for lookup
    offset_to_chunk = {v: k for k, v in chunk_offsets.items()}
    
    def mapper(fine_idx: int) -> int:
        fine_row, fine_col = fine_grid.index_to_row_col(fine_idx)
        
        # Which chunk is this in?
        chunk_row_idx = fine_row // chunk_rows
        chunk_col_idx = fine_col // chunk_cols
        
        # Find corresponding chunk key
        row_offset = chunk_row_idx * chunk_rows
        col_offset = chunk_col_idx * chunk_cols
        
        chunk_key = offset_to_chunk.get((row_offset, col_offset))
        if chunk_key is None:
            return -1
        
        q, r, s = chunk_key
        chunk_center = HexPosition(q, r, s) * (cover.rings * 2)
        
        # Local position within chunk
        local_row = fine_row - row_offset
        local_col = fine_col - col_offset
        
        # Scale down to coarse
        coarse_local_row = local_row // scale
        coarse_local_col = local_col // scale
        
        # Convert to coarse grid index
        # The chunk center in coarse grid
        center_idx = coarse_grid.hexposition_to_index(chunk_center, origin_index=0)
        if center_idx < 0:
            return -1
        
        center_row, center_col = coarse_grid.index_to_row_col(center_idx)
        
        # Offset from chunk center
        half_chunk = cover.rings
        coarse_row = center_row - half_chunk + coarse_local_row
        coarse_col = center_col - half_chunk + coarse_local_col
        
        return coarse_grid.row_col_to_index(coarse_row, coarse_col)
    
    return mapper





Can you show an example where you zoom in on a watershed in the aussie_map?

#| export
def stitch_chunks(chunks: set[ChunkRef], cover: ChunkCover, 
                  storage: GeoStorage, scale: int) -> tuple[Terrain, dict[tuple, tuple], int, int]:
    """Load chunks and stitch into single terrain.
    
    Returns (merged_terrain, chunk_offsets, chunk_rows, chunk_cols).
    """
    if not chunks:
        return None, {}, 0, 0
    
    coarse_grid = cover.terrain.hexGrid
    
    chunk_data = {}
    for chunk in chunks:
        origin_idx = coarse_grid.hexposition_to_index(chunk.center_hex, origin_index=0)
        if origin_idx < 0:
            continue
        result = storage.load_or_generate_chunk(cover, origin=origin_idx, scale=scale)
        if result.data is not None:
            coarse_row, coarse_col = coarse_grid.index_to_row_col(origin_idx)
            chunk_data[chunk.key] = (result.data, coarse_row, coarse_col)
    
    if not chunk_data:
        return None, {}, 0, 0
    
    sample_terrain = next(iter(chunk_data.values()))[0]
    chunk_rows = sample_terrain.hexGrid.nRows
    chunk_cols = sample_terrain.hexGrid.nCols
    chunk_radius = sample_terrain.hexGrid.radius
    
    coarse_rows = [d[1] for d in chunk_data.values()]
    coarse_cols = [d[2] for d in chunk_data.values()]
    min_coarse_row, max_coarse_row = min(coarse_rows), max(coarse_rows)
    min_coarse_col, max_coarse_col = min(coarse_cols), max(coarse_cols)
    
    max_row_spread = (max_coarse_row - min_coarse_row) * scale
    max_col_spread = (max_coarse_col - min_coarse_col) * scale
    
    merged_rows = max_row_spread + chunk_rows
    merged_cols = max_col_spread + chunk_cols
    
    merged_grid = HexGrid(
        nRows=merged_rows, nCols=merged_cols, 
        radius=chunk_radius, style=sample_terrain.hexGrid.style
    )
    
    merged_terrain = Terrain(bounds=merged_grid.bounds, radius=chunk_radius)
    merged_terrain.hexGrid = merged_grid
    merged_terrain.elevations = np.full(merged_rows * merged_cols, -100.0)
    merged_terrain.fields = {}
    
    for field_name in sample_terrain.fields:
        merged_terrain.fields[field_name] = np.zeros(merged_rows * merged_cols)
    
    if hasattr(sample_terrain, 'climate'):
        merged_terrain.climate = sample_terrain.climate
    if hasattr(sample_terrain, 'geo'):
        merged_terrain.geo = sample_terrain.geo
    
    chunk_offsets = {}
    
    for key, (chunk_terrain, coarse_row, coarse_col) in chunk_data.items():
        row_offset = (coarse_row - min_coarse_row) * scale
        col_offset = (coarse_col - min_coarse_col) * scale
        chunk_offsets[key] = (row_offset, col_offset)
        
        chunk_grid = chunk_terrain.hexGrid
        
        for local_idx in range(len(chunk_terrain.elevations)):
            local_row, local_col = chunk_grid.index_to_row_col(local_idx)
            
            merged_row = row_offset + local_row
            merged_col = col_offset + local_col
            
            if 0 <= merged_row < merged_rows and 0 <= merged_col < merged_cols:
                merged_idx = merged_row * merged_cols + merged_col
                merged_terrain.elevations[merged_idx] = chunk_terrain.elevations[local_idx]
                for field_name, field_data in chunk_terrain.fields.items():
                    merged_terrain.fields[field_name][merged_idx] = field_data[local_idx]
    
    return merged_terrain, chunk_offsets, chunk_rows, chunk_cols


In [ ]:
#| export
def build_fine_to_coarse_mapper(cover: ChunkCover, 
                                merged_terrain: Terrain,
                                chunk_offsets: dict,
                                scale: int,
                                chunk_rows: int,
                                chunk_cols: int) -> Callable[[int], int]:
    """Build function mapping fine terrain index → coarse terrain index."""
    coarse_grid = cover.terrain.hexGrid
    fine_grid = merged_terrain.hexGrid
    
    # Invert chunk_offsets for lookup
    offset_to_chunk = {v: k for k, v in chunk_offsets.items()}
    
    def mapper(fine_idx: int) -> int:
        fine_row, fine_col = fine_grid.index_to_row_col(fine_idx)
        
        # Find which chunk this fine hex belongs to
        best_key = None
        for (row_off, col_off), key in offset_to_chunk.items():
            if row_off <= fine_row < row_off + chunk_rows and \
               col_off <= fine_col < col_off + chunk_cols:
                best_key = key
                local_row = fine_row - row_off
                local_col = fine_col - col_off
                break
        
        if best_key is None:
            return -1
        
        # Scale down to coarse local position
        coarse_local_row = local_row // scale
        coarse_local_col = local_col // scale
        
        # Chunk center in coarse grid
        q, r, s = best_key
        chunk_center = HexPosition(q, r, s) * (cover.rings * 2)
        center_idx = coarse_grid.hexposition_to_index(chunk_center, origin_index=0)
        if center_idx < 0:
            return -1
        
        center_row, center_col = coarse_grid.index_to_row_col(center_idx)
        
        # The chunk terrain is centered on center_idx, so offset from top-left
        half_rows = chunk_rows // (2 * scale)
        half_cols = chunk_cols // (2 * scale)
        
        coarse_row = center_row - half_rows + coarse_local_row
        coarse_col = center_col - half_cols + coarse_local_col
        
        return coarse_grid.row_col_to_index(coarse_row, coarse_col)
    
    return mapper


#| export
@patch
def zoom_region(self: ChunkCover, 
                region: HexRegion,
                scale: int = 2,
                compute_weather: bool = True) -> ZoomResult:
    """Zoom into a region with stitched chunks and projected watersheds."""
    if self.db is None:
        raise ValueError("ChunkCover.db must be set for zoom_region")
    
    # Step 1: Get bounding box with padding
    min_row, max_row, min_col, max_col = region_bounding_box(region)
    
    padding = self.halo_rings + 1
    min_row = max(0, min_row - padding)
    max_row = min(self.terrain.hexGrid.nRows - 1, max_row + padding)
    min_col = max(0, min_col - padding)
    max_col = min(self.terrain.hexGrid.nCols - 1, max_col + padding)
    
    # Step 2: Find chunks
    chunks = bbox_to_chunk_refs(min_row, max_row, min_col, max_col, self)
    
    # Step 3: Load and stitch
    merged_terrain, chunk_offsets, chunk_rows, chunk_cols = stitch_chunks(
        chunks, self, self.db, scale
    )
    
    if merged_terrain is None:
        return ZoomResult(None, None, lambda x: -1, 0)
    
    fine_grid = merged_terrain.hexGrid
    
    # Step 4: Lazy init coarse basin
    if self.basin is None:
        self.basin = DrainageBasins(self.terrain)
        self.save()
    
    # Step 5: Build mapper BEFORE marking invalid
    mapper = build_fine_to_coarse_mapper(
        self, merged_terrain, chunk_offsets, scale,
        chunk_rows=chunk_rows, chunk_cols=chunk_cols
    )
    
    # Step 6: Mark invalid using mapper + elevation sentinel
    for fine_idx in range(len(merged_terrain.elevations)):
        coarse_idx = mapper(fine_idx)
        if coarse_idx < 0:
            fine_grid.invalidRegion.add(fine_idx)
        elif merged_terrain.elevations[fine_idx] <= -99.0:
            # Unset hexes from stitch (chunk's original invalidRegion)
            fine_grid.invalidRegion.add(fine_idx)
    
    # Step 7: Weather (after invalid marking so it can skip them)
    if compute_weather and hasattr(self.terrain, 'climate'):
        merged_terrain.climate = self.terrain.climate
        merged_terrain.geo = getattr(self.terrain, 'geo', None)
        merged_terrain.compute_weather()
    
    # Step 8: Build fine_regions for watershed projection
    fine_regions = {}
    for fine_idx in range(len(merged_terrain.elevations)):
        if fine_idx in fine_grid.invalidRegion:
            continue
        coarse_idx = mapper(fine_idx)
        if coarse_idx >= 0:
            if coarse_idx not in fine_regions:
                fine_regions[coarse_idx] = set()
            fine_regions[coarse_idx].add(fine_idx)
    
    # Step 9: Project watersheds
    zoomed_sheds = self.project_watersheds(self.basin, fine_regions, merged_terrain)
    
    zoomed_basins = DrainageBasins(merged_terrain, compute=False)
    zoomed_basins.sheds = zoomed_sheds
    
    return ZoomResult(
        terrain=merged_terrain,
        basins=zoomed_basins,
        mapper=mapper,
        chunks_loaded=len(chunk_offsets)
    )


In [ ]:
SVGBuilder.BUILDERHIDE = True

# Create terrain and cover with database
with GeoStorageDebugger(keep_on_error=True) as dbg:
    # 1. Load Sydney terrain
    terrain = TerraDemo().bayArea_map()
    terrain.compute_weather()
    
    print(f"Terrain: {terrain.hexGrid.nRows}x{terrain.hexGrid.nCols} = {len(terrain.hexGrid.hexes)} hexes")
    
    # 2. Create cover and attach DB
    cover = ChunkCover(terrain, rings=5, halo_rings=1)
    cover.db = dbg.server
    cover.save(name="Sydney Test")
    
    # 3. Compute basins on coarse terrain
    basins = DrainageBasins(terrain)
    cover.basin = basins
    
    print(f"Found {len(basins.sheds)} watersheds")
    
    # 4. Pick a watershed and make a HexRegion
    # Find a medium-sized one (not too big, not too small)
    shed_sizes = [(i, len(ws.region.hexes)) for i, ws in enumerate(basins.sheds)]
    shed_sizes.sort(key=lambda x: x[1], reverse=True)
    
    # Pick the 3rd largest
    target_idx = shed_sizes[2][0] if len(shed_sizes) > 2 else 0
    target_shed = basins.sheds[target_idx]
    
    print(f"Selected watershed {target_idx}: {len(target_shed.region.hexes)} hexes")
    els =  [terrain.elevations[i] for i in target_shed.region.hexes]
    print(els)
    
    # Convert Watershed hexes to HexRegion
    region = target_shed.region
    
    # 5. Zoom into the region
    result = cover.zoom_region(region, scale=2, compute_weather=True)
    
    print(f"\nZoom result:")
    print(f"  Chunks loaded: {result.chunks_loaded}")
    print(f"  Fine terrain: {result.terrain.hexGrid.nRows}x{result.terrain.hexGrid.nCols}")
    print(f"  Watersheds: {len(result.basins.sheds)}")
    for i in range(len(terrain.elevations)):
        if i not in region:
            terrain.elevations[i] = 3000
    terrain.textElevations()
    print("now detail")


    
    # 6. Visualize
    result.terrain.colorMap()
    result.terrain.hexGrid.update()
    result.terrain.textElevations()
    terrain = result.terrain
    print(f"result Terrain: {terrain.hexGrid.nRows}x{terrain.hexGrid.nCols} = {len(terrain.hexGrid.hexes)} hexes")
result.terrain.hexGrid.builder.show()


Something broke with getting zoomed

Please write the corrected zoom_region

In [ ]:
SVGBuilder.BUILDERHIDE = True

In [ ]:

# Create terrain and cover with database
with GeoStorageDebugger(keep_on_error=True) as dbg:
    # 1. Load Sydney terrain
    terrain = TerraDemo().bayArea_map()
    terrain.compute_weather()
    
    print(f"Terrain: {terrain.hexGrid.nRows}x{terrain.hexGrid.nCols} = {len(terrain.hexGrid.hexes)} hexes")
    
    # 2. Create cover and attach DB
    cover = ChunkCover(terrain, rings=5, halo_rings=1)
    cover.db = dbg.server
    cover.save(name="Sydney Test")
    
    # 3. Compute basins on coarse terrain
    basins = DrainageBasins(terrain)
    cover.basin = basins
    
    print(f"Found {len(basins.sheds)} watersheds")
    
    # 4. Pick a watershed and make a HexRegion
    shed_sizes = [(i, len(ws.region.hexes)) for i, ws in enumerate(basins.sheds)]
    shed_sizes.sort(key=lambda x: x[1], reverse=True)
    
    target_idx = shed_sizes[2][0] if len(shed_sizes) > 2 else 0
    target_shed = basins.sheds[target_idx]
    region = target_shed.region
    
    print(f"Selected watershed {target_idx}: {len(region.hexes)} hexes")
    
    # 5. Show the coarse map with the region highlighted
    terrain.colorMap()
    
    # Highlight the selected region
    highlight_style = StyleCSS("highlight", fill="#ff000088", stroke="#ff0000", stroke_width=2)
    terrain.hexGrid.builder.add_style(highlight_style)
    
    for idx in region.hexes:
        terrain.hexGrid.hexes[idx].style = highlight_style
    
    terrain.hexGrid.update()
    
print("Coarse terrain with highlighted region:")
terrain.hexGrid.builder.show()


kingdom_detail should use Chunk_Cover.zoom_region for its watershed/ basins

should I add __mul__ to HexPosition

## Real Data

## Code Quality Analysis

### ✅ Strengths

1. **Comprehensive caching:** Terrain, weather, watersheds all cached
2. **Smart cache keys:** Using (cover_id, origin_pos, scale) uniquely identifies chunks
3. **Automatic invalidation:** Ensures consistency when coarse data changes
4. **Position-based queries:** Using HexPosition prevents grid alignment issues
5. **Efficient watershed projection:** O(n×k²) not O(n²)
6. **Result objects:** Consistent LoadResult/SaveResult pattern
7. **Testing infrastructure:** GeoStorageDebugger for test isolation

### ⚠️ Concerns

1. **Cache Size Management**
   - No LRU eviction
   - No max cache size limit
   - Could grow unbounded in long-running apps

2. **Chunk Reconstruction**
   - `_reconstruct_terrain_from_rows()` infers grid dimensions from data
   - Could fail if chunk has irregular shape
   - No validation of reconstructed grid

3. **Transaction Safety**
   - Some operations use `with self.db.conn:` (good)
   - Others don't (potential partial writes)
   - Invalidation is multi-statement (not atomic)

4. **Error Handling**
   - Generic `except Exception as e:` catches everything
   - Could hide bugs (e.g., programming errors vs. data errors)
   - No retry logic for transient failures

5. **Missing Features**
   - No access time tracking (needed for LRU)
   - No cache statistics (hit rate, size, age)
   - No prefetching hints
   - No background cache warming

---

## Suggestions for Improvement

### 1. Cache Management

#### Add LRU Eviction
```python
@dataclass
class CacheStats:
    cover_id: int
    chunk_count: int
    total_hexes: int
    oldest_access: int
    newest_access: int
    total_size_mb: float

@patch
def get_cache_stats(self: GeoStorage, cover_id: int = None) -> CacheStats:
    """Get statistics about cached chunks."""
    # Count chunks, hexes, compute size
    # Track access times (need to add last_accessed column)

@patch
def evict_lru_chunks(self: GeoStorage, cover_id: int, 
                     target_size_mb: float = 100) -> SaveResult:
    """Evict least-recently-accessed chunks until under size limit."""
    # Requires: ALTER TABLE hex_data ADD COLUMN last_accessed INTEGER
    # Query chunks by access time
    # Delete oldest until size < target
```

#### Add Access Tracking
```python
# Schema change needed:
@dataclass
class HexData:
    # ... existing ...
    last_accessed: int = 0  # NEW: Unix timestamp
    access_count: int = 0   # NEW: For popularity tracking

# Update on load:
@patch
def _load_cached_chunk(self: GeoStorage, ...):
    # ... load data ...
    
    # Update access stats
    now = int(datetime.now().timestamp())
    self.db.execute("""
        UPDATE hex_data 
        SET last_accessed = ?, access_count = access_count + 1
        WHERE world_id = ? AND chunk_q = ? AND chunk_r = ? AND chunk_s = ?
          AND scale_level = ?
    """, [now, cover_id, origin_pos.q, origin_pos.r, origin_pos.s, scale])
```

### 2. Chunk Reconstruction Safety

#### Validate Reconstructed Grid
```python
@patch
def _reconstruct_terrain_from_rows(self: GeoStorage, rows: list[dict], 
                                   cover_id: int) -> Terrain:
    # ... existing logic ...
    
    # VALIDATE reconstructed grid
    expected_hex_count = len(rows)
    actual_hex_count = len(grid.hexes)
    
    if actual_hex_count != expected_hex_count:
        raise ValueError(
            f"Grid reconstruction mismatch: {actual_hex_count} grid hexes "
            f"vs {expected_hex_count} data rows. Chunk may be incomplete."
        )
    
    # Verify no gaps in coverage
    positions_in_rows = {(r['q'], r['r'], r['s']) for r in rows}
    positions_in_grid = {
        (grid.index_to_hexposition(i).q, 
         grid.index_to_hexposition(i).r,
         grid.index_to_hexposition(i).s)
        for i in range(len(grid.hexes))
    }
    
    missing = positions_in_grid - positions_in_rows
    if missing:
        print(f"⚠ Warning: {len(missing)} hexes missing data (will be zero)")
```

#### Store Grid Metadata
```python
# Alternative: Store grid params explicitly

@dataclass
class CachedChunk:
    """Metadata for cached chunks."""
    id: int = None
    cover_id: int = 0
    chunk_q, chunk_r, chunk_s: int
    scale: int = 2
    # Store grid reconstruction params
    nrows: int = 0
    ncols: int = 0
    radius: float = 0.0
    hex_count: int = 0
    created, last_accessed, access_count: int

# Then use stored params for reconstruction
```

### 3. Transaction Safety

#### Wrap Multi-Statement Operations
```python
@patch
def invalidate_all_caches(self: GeoStorage, cover_id: int) -> dict:
    """Invalidate all cached data when coarse terrain changes."""
    results = {}
    
    # Use transaction for atomic invalidation
    try:
        with self.db.conn:
            # All deletes in one transaction
            cursor = self.db.execute(
                "DELETE FROM hex_data WHERE world_id = ? AND scale_level > 0",
                [cover_id]
            )
            results['terrain'] = SaveResult(cursor.rowcount, 'invalidated', ...)
            
            cursor = self.db.execute(
                "DELETE FROM hex_weather WHERE world_id = ? AND scale_level > 0",
                [cover_id]
            )
            results['weather'] = SaveResult(cursor.rowcount, 'invalidated', ...)
            
            # ... more deletes ...
            
        return results
    
    except Exception as e:
        # Transaction rolled back automatically
        return {'error': SaveResult(None, 'error', str(e))}
```

### 4. Error Handling Refinement

#### Distinguish Error Types
```python
class CacheError(Exception):
    """Recoverable cache errors."""

class DataIntegrityError(Exception):
    """Unrecoverable data corruption."""

@patch
def load_or_generate_chunk(self: GeoStorage, ...):
    try:
        # Check cache
        cached = self._load_cached_chunk(...)
        if cached.status == 'loaded':
            return LoadResult(cached.data, 'cached', cached.context)
    
    except DataIntegrityError as e:
        # Log and invalidate bad cache entry
        print(f"⚠ Cache corruption detected: {e}")
        self.invalidate_chunk_cache(cover.ident)
        # Fall through to regeneration
    
    except CacheError as e:
        # Transient error, could retry
        print(f"⚠ Cache lookup failed: {e}")
        # Fall through to regeneration
    
    # Generate fresh
    try:
        zoomed = cover.zoomChunkCombined(...)
        self._save_chunk_cache(...)
        return LoadResult(zoomed, 'generated', ...)
    
    except Exception as e:
        # Generation failed - this is serious
        return LoadResult(None, 'error', f'Generation failed: {e}')
```

### 5. Background Cache Operations

#### Prefetch Adjacent Chunks
```python
@patch
def prefetch_neighbors(self: GeoStorage, cover: ChunkCover, 
                       origin: int, scale: int) -> list[SaveResult]:
    """Background prefetch of neighboring chunks."""
    grid = cover.terrain.hexGrid
    origin_pos = grid.index_to_hexposition(origin)
    
    neighbors = []
    for direction in HexPosition.directions():
        neighbor_origin = origin_pos + (direction * cover.distance)
        neighbor_idx = grid.hexposition_to_index(neighbor_origin, origin_index=0)
        
        if neighbor_idx >= 0:
            neighbors.append((neighbor_idx, neighbor_origin))
    
    # Generate in background (could use ThreadPoolExecutor)
    results = []
    for idx, pos in neighbors:
        if not self.has_cached_chunk(cover.ident, pos, scale):
            result = self.load_or_generate_chunk(cover, idx, scale)
            results.append(result)
    
    return results
```

### 6. Cache Statistics Dashboard

#### Implement Monitoring
```python
@dataclass
class CacheMetrics:
    cover_id: int
    total_chunks: int
    total_hexes: int
    terrain_size_mb: float
    weather_size_mb: float
    watershed_size_mb: float
    hit_rate: float         # hits / (hits + misses)
    avg_age_hours: float
    oldest_chunk_age_hours: float

@patch
def get_cache_metrics(self: GeoStorage, cover_id: int) -> CacheMetrics:
    """Comprehensive cache statistics."""
    # Query chunk counts, sizes, access patterns
    # Compute hit rate from access_count and generation timestamps
    # Calculate ages from created/last_accessed

@patch
def print_cache_report(self: GeoStorage, cover_id: int):
    """Human-readable cache report."""
    metrics = self.get_cache_metrics(cover_id)
    
    print(f"Cache Report for Cover {cover_id}")
    print(f"  Chunks: {metrics.total_chunks} ({metrics.total_hexes} hexes)")
    print(f"  Size: {metrics.terrain_size_mb + metrics.weather_size_mb:.1f} MB")
    print(f"    - Terrain: {metrics.terrain_size_mb:.1f} MB")
    print(f"    - Weather: {metrics.weather_size_mb:.1f} MB")
    print(f"    - Watersheds: {metrics.watershed_size_mb:.1f} MB")
    print(f"  Hit Rate: {metrics.hit_rate:.1%}")
    print(f"  Age: avg {metrics.avg_age_hours:.1f}h, max {metrics.oldest_chunk_age_hours:.1f}h")
```

---

## Documentation Updates Needed

### 1. Update DatabaseSchema.md

Add sections:
- ✅ **ChunkCover Integration** - save/load, persistence
- ✅ **Chunk Caching** - cache keys, hit rates, invalidation
- ✅ **Weather Caching** - seasonal caching per chunk
- ✅ **Watershed Persistence** - assignment caching, projection
- ✅ **New Tables** - WatershedMeta, ChunkBorder
- ✅ **Schema Extensions** - updated HexData, HexWeather fields

### 2. Update DATABASE_QUICK_REF.md

Add examples:
```python
# ChunkCover workflow
cover = ChunkCover(master, rings=5)
cover.db = storage
cover.save(name="Master")

# Cached zooming
zoomed = cover.zoom_cached(origin=middle, scale=4)  # Generates + caches
zoomed2 = cover.zoom_cached(origin=middle, scale=4) # Cache hit!

# Full data zoom (terrain + weather + watersheds)
full = cover.zoom_with_full_data(origin=middle, scale=4)

# Cache management
storage.invalidate_all_caches(cover.ident)
stats = storage.get_cache_stats(cover.ident)
```

Good news: the `deepcopy`/`combine_rivers` nightmare is gone (was 47s, now 0). Bad news: it's still 25s because of **chunk decoding**. Here's where the time goes:

| Time | What | Why |
|------|------|-----|
| **14.5s** | `_build_hexes` (62 calls) | Each chunk creates a full HexGrid with `round()` on every hex |
| **7.4s** | `soil.from_plates` (2197 calls) | Rebuilding watershed soil data inside `basin.decode` |
| **5.3s** | `round()` (1.7M calls) | Inside `_build_hexes` computing hex centers |
| **4.8s** | `Hex.__init__` (423K calls) | Creating hex objects for all 20 chunks |
| **2.3s** | `fetchall` from DB | Actually reading the chunk rows |

So 20 chunks are loaded, each one fully reconstructs its own `Terrain` → `HexGrid` → all `Hex` objects → all `DrainageBasins` → all `Watershed` objects from the DB rows. Then they get stitched together and most of those intermediate objects are thrown away.

The question is: **do you need the full hex grids and basins for each individual chunk during stitching, or could you decode just the field data (elevations, country, etc.) and build the hex grid once at the end on the stitched result?**

That would mean a lightweight `decode_fields_only()` path that skips `_build_hexes` and `basin.decode` per chunk, then constructs the grid and projects basins only once on the final merged terrain. That alone could cut this from 25s to ~3-4s.

I think we need a better flow

I think a dataproxy class is going to be helpful, but yes in general this seems like good thinking

#| export
class DataProxy:
    """Lightweight chunk data holder for efficient stitching.
    
    Holds only raw numpy arrays without HexGrid geometry.
    """
    nRows: int
    nCols: int
    radius: float
    elevations: np.ndarray  # Shape: (nRows * nCols,)
    watersheds: Optional[np.ndarray] = None  # Shape: (nRows * nCols,), watershed IDs
    chunk_position: Optional[HexPosition] = None  # For debugging/tracking
    invalidRegion: set[int] = field(default_factory=set)  # Track invalid hexes
    
    # Metadata for validation
    expected_count: int = 0  # How many valid hexes we expect
    actual_count: int = 0    # How many we actually loaded
    
    @property
    def size(self) -> int:
        return self.nRows * self.nCols
    
    def __len__(self):
        return self.size
    
    @property
    def is_complete(self) -> bool:
        """Check if all expected hexes were loaded."""
        if self.expected_count == 0:
            return True  # No expectation set
        return self.actual_count >= self.expected_count * 0.95  # Allow 5% tolerance
    
    def to_dict(self) -> dict:
        """Convert to dict for debugging."""
        return {
            'nRows': self.nRows,
            'nCols': self.nCols,
            'radius': self.radius,
            'size': self.size,
            'has_watersheds': self.watersheds is not None,
            'chunk_position': str(self.chunk_position) if self.chunk_position else None,
            'invalid_count': len(self.invalidRegion),
            'complete': self.is_complete,
            'expected': self.expected_count,
            'actual': self.actual_count
        }

#| export
@patch
def _load_chunk_as_proxy(self: GeoStorage, 
                         cover_id: int,
                         origin_pos: HexPosition,
                         scale: int) -> LoadResult:
    """Load cached chunk as DataProxy (fully numpy-optimized)."""
    try:
        # Get cover metadata first (outside transaction)
        world = self.worlds[cover_id]
        cover_data = GeoStorage._get(world, 'cover_data')
        if not cover_data:
            return LoadResult(None, 'error', 'No cover data found')
        
        original_cover = ChunkCover.decode(cover_data)
        base_radius = original_cover.terrain.hexGrid.radius
        radius = base_radius / scale
        
        # Wrap SELECT + UPDATE in transaction — extract cols/rows INSIDE
        with self.db.conn:
            cursor = self.db.execute("""
                SELECT h.* FROM hex_data h
                INNER JOIN (
                    SELECT world_id, q, r, s, MAX(modified) as max_mod
                    FROM hex_data
                    WHERE world_id = ? 
                      AND chunk_q = ? AND chunk_r = ? AND chunk_s = ?
                      AND scale_level = ?
                    GROUP BY world_id, q, r, s
                ) latest 
                    ON h.world_id = latest.world_id 
                    AND h.q = latest.q AND h.r = latest.r AND h.s = latest.s
                    AND h.modified = latest.max_mod
            """, [cover_id, origin_pos.q, origin_pos.r, origin_pos.s, scale])
            
            # Must extract INSIDE transaction before cursor closes
            cols = [d[0] for d in cursor.description]
            rows = [dict(zip(cols, row)) for row in cursor.fetchall()]
            
            if not rows:
                return LoadResult(None, 'not_found', 'No cached chunk')
            
            # Update access stats (still in transaction)
            now = int(datetime.now().timestamp())
            self.db.execute("""
                UPDATE hex_data 
                SET last_accessed = ?, access_count = access_count + 1
                WHERE world_id = ? AND chunk_q = ? AND chunk_r = ? AND chunk_s = ?
                AND scale_level = ?
            """, [now, cover_id, origin_pos.q, origin_pos.r, origin_pos.s, scale])
        
        # --- Everything below is pure numpy, no DB ---
        
        # Infer grid dimensions
        qs = np.array([r['q'] for r in rows])
        rs = np.array([r['r'] for r in rows])
        ss = np.array([r['s'] for r in rows])
        
        min_q, max_q = int(qs.min()), int(qs.max())
        min_r, max_r = int(rs.min()), int(rs.max())
        
        nRows = max_r - min_r + 1
        nCols = max_q - min_q + 1
        
        # Initialize arrays
        elevations = np.full(nRows * nCols, -100.0)
        watersheds = np.full(nRows * nCols, -1, dtype=int)
        
        # Check if we have usable grid_index
        has_grid_index = all(
            'grid_index' in row and row['grid_index'] is not None 
            and 0 <= row['grid_index'] < nRows * nCols
            for row in rows
        )
        
        if has_grid_index:
            # Fast path: stored indices
            indices = np.array([row['grid_index'] for row in rows], dtype=int)
        else:
            # Vectorized coordinate conversion via temp grid
            positions = np.column_stack([
                qs - min_q,
                rs - min_r,
                ss - (-min_q - min_r)  # Correct s offset
            ])
            
            temp_grid = HexGrid(
                nRows=nRows, nCols=nCols, radius=radius,
                style=StyleCSS("simple")
            )
            indices = temp_grid.hexpositions_to_indices(positions, origin_index=0)
        
        # Extract data as numpy arrays
        elev_values = np.array([row['elevation'] for row in rows])
        ws_values = np.array([
            row.get('watershed_id') if row.get('watershed_id') is not None else -1 
            for row in rows
        ], dtype=int)
        
        # Filter valid indices
        valid_mask = (indices >= 0) & (indices < nRows * nCols)
        valid_indices = indices[valid_mask]
        valid_elevs = elev_values[valid_mask]
        valid_ws = ws_values[valid_mask]
        
        # Vectorized assignment
        elevations[valid_indices] = valid_elevs
        watersheds[valid_indices] = valid_ws
        
        # Mark invalid hexes (sentinel elevation)
        water_mask = valid_elevs <= -99.0
        invalidRegion = set(valid_indices[water_mask].tolist())
        
        # Count valid hexes
        actual_count = int(np.sum(elevations > -99.0))
        chunk_rings = original_cover.rings
        expected_count = 3 * chunk_rings * chunk_rings
        
        proxy = DataProxy(
            nRows=nRows,
            nCols=nCols,
            radius=radius,
            elevations=elevations,
            watersheds=watersheds,
            chunk_position=origin_pos,
            invalidRegion=invalidRegion,
            expected_count=expected_count,
            actual_count=actual_count
        )
        
        if not proxy.is_complete:
            return LoadResult(proxy, 'partial', 
                            f'{actual_count}/{expected_count} hexes (incomplete)')
        
        return LoadResult(proxy, 'loaded', f'{len(rows)} hexes as DataProxy')
    
    except Exception as e:
        return LoadResult(None, 'error', str(e))


#| export
def stitch_chunks_fast(chunks: set[ChunkRef], 
                       cover: ChunkCover,
                       storage: GeoStorage, 
                       scale: int) -> tuple[DataProxy, dict[tuple, tuple], int, int]:
    """Load chunks as DataProxy and stitch using numpy block copies.
    
    Generates chunks on cache miss, then loads as lightweight proxy.
    Returns (stitched_proxy, chunk_offsets, chunk_rows, chunk_cols).
    """
    if not chunks:
        return None, {}, 0, 0
    
    coarse_grid = cover.terrain.hexGrid
    
    # Load (or generate) all chunks as DataProxy
    chunk_proxies = {}
    for chunk in chunks:
        origin_idx = coarse_grid.hexposition_to_index(chunk.center_hex, origin_index=0)
        if origin_idx < 0:
            continue  # Chunk center outside grid, skip
        
        # Try loading from cache first
        result = storage._load_chunk_as_proxy(cover.ident, chunk.position, scale)
        
        if result.status == 'not_found':
            # Cache miss — generate the chunk, then load as proxy
            gen = storage.load_or_generate_chunk(cover, origin=origin_idx, scale=scale)
            if gen.status in ('generated', 'cached'):
                result = storage._load_chunk_as_proxy(cover.ident, chunk.position, scale)
            else:
                print(f"⚠️  Failed to generate chunk {chunk.key}: {gen.context}")
                continue
        
        if result.status == 'error':
            print(f"⚠️  Error loading chunk {chunk.key}: {result.context}")
            continue
        
        if result.status in ('loaded', 'partial'):
            coarse_row, coarse_col = coarse_grid.index_to_row_col(origin_idx)
            chunk_proxies[chunk.key] = (result.data, coarse_row, coarse_col)
    
    if not chunk_proxies:
        return None, {}, 0, 0
    
    # Validate dimensions — all chunks should be same size
    first_proxy = next(iter(chunk_proxies.values()))[0]
    chunk_rows = first_proxy.nRows
    chunk_cols = first_proxy.nCols
    radius = first_proxy.radius
    
    mismatched = {k for k, (p, _, _) in chunk_proxies.items() 
                  if p.nRows != chunk_rows or p.nCols != chunk_cols}
    if mismatched:
        print(f"⚠️  Dimension mismatch in chunks: {mismatched}, skipping them")
        for k in mismatched:
            del chunk_proxies[k]
    
    if not chunk_proxies:
        return None, {}, 0, 0
    
    # Calculate merged dimensions from coarse positions
    coarse_rows = [d[1] for d in chunk_proxies.values()]
    coarse_cols = [d[2] for d in chunk_proxies.values()]
    min_coarse_row, max_coarse_row = min(coarse_rows), max(coarse_rows)
    min_coarse_col, max_coarse_col = min(coarse_cols), max(coarse_cols)
    
    merged_rows = (max_coarse_row - min_coarse_row) * scale + chunk_rows
    merged_cols = (max_coarse_col - min_coarse_col) * scale + chunk_cols
    
    # Allocate merged 2D arrays
    merged_elev = np.full((merged_rows, merged_cols), -100.0)
    merged_ws = np.full((merged_rows, merged_cols), -1, dtype=int)
    merged_invalid = set()
    
    chunk_offsets = {}
    
    # Block-copy each chunk
    for key, (proxy, coarse_row, coarse_col) in chunk_proxies.items():
        row_off = (coarse_row - min_coarse_row) * scale
        col_off = (coarse_col - min_coarse_col) * scale
        chunk_offsets[key] = (row_off, col_off)
        
        # Reshape proxy to 2D
        p_elev = proxy.elevations.reshape(proxy.nRows, proxy.nCols)
        p_ws = proxy.watersheds.reshape(proxy.nRows, proxy.nCols)
        
        # Clip source/dest ranges
        sr0, sr1 = 0, proxy.nRows
        sc0, sc1 = 0, proxy.nCols
        dr0, dr1 = row_off, row_off + proxy.nRows
        dc0, dc1 = col_off, col_off + proxy.nCols
        
        if dr0 < 0:          sr0 -= dr0; dr0 = 0
        if dr1 > merged_rows: sr1 -= (dr1 - merged_rows); dr1 = merged_rows
        if dc0 < 0:          sc0 -= dc0; dc0 = 0
        if dc1 > merged_cols: sc1 -= (dc1 - merged_cols); dc1 = merged_cols
        
        if dr0 < dr1 and dc0 < dc1:
            merged_elev[dr0:dr1, dc0:dc1] = p_elev[sr0:sr1, sc0:sc1]
            merged_ws[dr0:dr1, dc0:dc1] = p_ws[sr0:sr1, sc0:sc1]
        
        # Remap invalidRegion with offset
        for local_idx in proxy.invalidRegion:
            lr = local_idx // proxy.nCols
            lc = local_idx % proxy.nCols
            mr = row_off + lr
            mc = col_off + lc
            if 0 <= mr < merged_rows and 0 <= mc < merged_cols:
                merged_invalid.add(mr * merged_cols + mc)
    
    merged_proxy = DataProxy(
        nRows=merged_rows,
        nCols=merged_cols,
        radius=radius,
        elevations=merged_elev.ravel(),
        watersheds=merged_ws.ravel(),
        invalidRegion=merged_invalid,
        expected_count=sum(p[0].expected_count for p in chunk_proxies.values()),
        actual_count=sum(p[0].actual_count for p in chunk_proxies.values())
    )
    
    return merged_proxy, chunk_offsets, chunk_rows, chunk_cols


#| export
def proxy_to_terrain(proxy: DataProxy, 
                     climate: ClimatePreset = None,
                     geo: GeoBounds = None) -> Terrain:
    """Convert DataProxy to full Terrain with geometry.
    
    Preserves invalidRegion from the proxy.
    """
    bounds = MapRect(
        MapCord(0, 0), 
        MapSize(proxy.nRows * proxy.radius, proxy.nCols * proxy.radius)
    )
    
    terrain = Terrain(
        bounds=bounds, 
        radius=proxy.radius,
        climate=climate,
        geo=geo
    )
    
    # Set grid dimensions
    terrain.hexGrid.nRows = proxy.nRows
    terrain.hexGrid.nCols = proxy.nCols
    terrain.hexGrid.adjustRadius(proxy.radius)
    
    # Copy arrays
    terrain.elevations = proxy.elevations.copy()
    
    # Add watershed field if present
    if proxy.watersheds is not None and np.any(proxy.watersheds >= 0):
        terrain.fields['watershed_id'] = proxy.watersheds.copy()
    
    # Preserve invalidRegion
    terrain.hexGrid.invalidRegion = proxy.invalidRegion.copy()
    
    return terrain


@patch
def validate_chunk_cache(self: GeoStorage, cover_id: int, 
                         origin_pos: HexPosition, 
                         scale: int) -> dict:
    """Validate integrity of cached chunk data.
    
    Returns dict with validation results.
    """
    result = {
        'valid': False,
        'hex_count': 0,
        'expected_count': 0,
        'has_watersheds': False,
        'has_gaps': False,
        'elevation_range': (0, 0)
    }
    
    try:
        cursor = self.db.execute("""
            SELECT COUNT(*) as count,
                   MIN(elevation) as min_elev,
                   MAX(elevation) as max_elev,
                   COUNT(DISTINCT watershed_id) as watershed_count
            FROM hex_data
            WHERE world_id = ? 
              AND chunk_q = ? AND chunk_r = ? AND chunk_s = ?
              AND scale_level = ?
        """, [cover_id, origin_pos.q, origin_pos.r, origin_pos.s, scale])
        
        row = cursor.fetchone()
        if row:
            result['hex_count'] = row[0]
            result['elevation_range'] = (row[1], row[2])
            result['has_watersheds'] = row[3] > 0
            
            # Get expected count from cover
            world = self.worlds[cover_id]
            cover_data = GeoStorage._get(world, 'cover_data')
            if cover_data:
                cover = ChunkCover.decode(cover_data)
                result['expected_count'] = 3 * cover.rings * cover.rings
                result['valid'] = result['hex_count'] >= result['expected_count'] * 0.95
                result['has_gaps'] = result['hex_count'] < result['expected_count']
        
        return result
    
    except Exception as e:
        result['error'] = str(e)
        return result

#| export
@patch
def zoom_region_with_proxy(self: ChunkCover, 
                            region: HexRegion,
                            scale: int = 2,
                            compute_weather: bool = True) -> ZoomResult:
    """Zoom into a region using DataProxy for efficient stitching (FIXED)."""
    if self.db is None:
        raise ValueError("ChunkCover.db must be set")
    
    # Step 1: Get bounding box with padding
    min_row, max_row, min_col, max_col = region_bounding_box(region)
    
    padding = self.halo_rings + 1
    min_row = max(0, min_row - padding)
    max_row = min(self.terrain.hexGrid.nRows - 1, max_row + padding)
    min_col = max(0, min_col - padding)
    max_col = min(self.terrain.hexGrid.nCols - 1, max_col + padding)
    
    # Step 2: Find chunks
    chunks = bbox_to_chunk_refs(min_row, max_row, min_col, max_col, self)
    
    # Step 3: Load and stitch as DataProxy (returns chunk dimensions)
    merged_proxy, chunk_offsets, chunk_rows, chunk_cols = stitch_chunks_fast(
        chunks, self, self.db, scale
    )
    
    if merged_proxy is None:
        return ZoomResult(None, None, lambda x: -1, 0)
    
    # Warn if incomplete
    if not merged_proxy.is_complete:
        print(f"⚠️  Merged terrain incomplete: {merged_proxy.actual_count}/{merged_proxy.expected_count} hexes")
    
    # Step 4: Build Terrain once from merged proxy (geometry computed here)
    merged_terrain = proxy_to_terrain(
        merged_proxy,
        climate=self.terrain.climate,
        geo=getattr(self.terrain, 'geo', None)
    )
    
    fine_grid = merged_terrain.hexGrid
    
    # Step 5: Lazy init coarse basin
    if self.basin is None:
        self.basin = DrainageBasins(self.terrain)
        self.save()
    
    # Step 6: Build mapper with CORRECT chunk dimensions
    mapper = build_fine_to_coarse_mapper(
        self, merged_terrain, chunk_offsets, scale,
        chunk_rows=chunk_rows,
        chunk_cols=chunk_cols
    )
    
    # Step 7: Validate and extend invalidRegion using mapper
    for fine_idx in range(len(merged_terrain.elevations)):
        if fine_idx in fine_grid.invalidRegion:
            continue  # Already marked by proxy
        
        coarse_idx = mapper(fine_idx)
        if coarse_idx < 0:
            fine_grid.invalidRegion.add(fine_idx)
        elif merged_terrain.elevations[fine_idx] <= -99.0:
            fine_grid.invalidRegion.add(fine_idx)
    
    # Step 8: Weather (after invalid marking)
    if compute_weather and hasattr(self.terrain, 'climate'):
        merged_terrain.climate = self.terrain.climate
        merged_terrain.geo = getattr(self.terrain, 'geo', None)
        merged_terrain.compute_weather()
    
    # Step 9: Build fine_regions for watershed projection
    fine_regions = {}
    for fine_idx in range(len(merged_terrain.elevations)):
        if fine_idx in fine_grid.invalidRegion:
            continue
        coarse_idx = mapper(fine_idx)
        if coarse_idx >= 0:
            if coarse_idx not in fine_regions:
                fine_regions[coarse_idx] = set()
            fine_regions[coarse_idx].add(fine_idx)
    
    # Step 10: Project watersheds
    zoomed_sheds = self.project_watersheds(self.basin, fine_regions, merged_terrain)
    
    zoomed_basins = DrainageBasins(merged_terrain, compute=False)
    zoomed_basins.sheds = zoomed_sheds
    
    return ZoomResult(
        terrain=merged_terrain,
        basins=zoomed_basins,
        mapper=mapper,
        chunks_loaded=len(chunk_offsets)
    )

@patch
def zoom_region_fast(self: ChunkCover, 
                     region: HexRegion,
                     scale: int = 2,
                     compute_weather: bool = True) -> ZoomResult:
    """Zoom into a region using DataProxy for efficient stitching.
    
    Projects weather/climate fields from coarse terrain using mapper.
    """
    if self.db is None:
        raise ValueError("ChunkCover.db must be set")
    
    # Step 1: Get bounding box with padding
    min_row, max_row, min_col, max_col = region_bounding_box(region)
    
    padding = self.halo_rings + 1
    min_row = max(0, min_row - padding)
    max_row = min(self.terrain.hexGrid.nRows - 1, max_row + padding)
    min_col = max(0, min_col - padding)
    max_col = min(self.terrain.hexGrid.nCols - 1, max_col + padding)
    
    # Step 2: Find chunks
    chunks = bbox_to_chunk_refs(min_row, max_row, min_col, max_col, self)
    
    # Step 3: Load and stitch as DataProxy
    merged_proxy, chunk_offsets, chunk_rows, chunk_cols = stitch_chunks_fast(
        chunks, self, self.db, scale
    )
    
    if merged_proxy is None:
        return ZoomResult(None, None, lambda x: -1, 0)
    
    # Warn if incomplete
    if not merged_proxy.is_complete:
        print(f"⚠️  Merged terrain incomplete: {merged_proxy.actual_count}/{merged_proxy.expected_count} hexes")
    
    # Step 4: Convert DataProxy to Terrain (builds HexGrid once)
    merged_terrain = proxy_to_terrain(
        merged_proxy,
        climate=getattr(self.terrain, 'climate', None),
        geo=getattr(self.terrain, 'geo', None)
    )
    
    fine_grid = merged_terrain.hexGrid
    
    # Step 5: Lazy init coarse basin
    if self.basin is None:
        self.basin = DrainageBasins(self.terrain)
        self.save()
    
    # Step 6: Build mapper
    mapper = build_fine_to_coarse_mapper(
        self, merged_terrain, chunk_offsets, scale,
        chunk_rows=chunk_rows,
        chunk_cols=chunk_cols
    )
    
    # Step 7: Validate and extend invalidRegion
    for fine_idx in range(len(merged_terrain.elevations)):
        if fine_idx in fine_grid.invalidRegion:
            continue  # Already marked by proxy
        
        coarse_idx = mapper(fine_idx)
        if coarse_idx < 0:
            fine_grid.invalidRegion.add(fine_idx)
        elif merged_terrain.elevations[fine_idx] <= -99.0:
            fine_grid.invalidRegion.add(fine_idx)
    
    # Step 8: Project weather fields from coarse terrain
    if compute_weather:
        coarse_terrain = self.terrain
        
        # Check which fields exist on coarse terrain
        weather_fields = ['temperature', 'precipitation', 'humidity', 
                         'climate_pet', 'aridity_index', 'climate']
        
        for field_name in weather_fields:
            if field_name in coarse_terrain.fields:
                # Initialize field on fine terrain
                merged_terrain.fields[field_name] = np.zeros(len(merged_terrain.elevations))
                
                # Project using mapper (vectorized where possible)
                for fine_idx in range(len(merged_terrain.elevations)):
                    if fine_idx in fine_grid.invalidRegion:
                        continue
                    
                    coarse_idx = mapper(fine_idx)
                    if coarse_idx >= 0 and coarse_idx < len(coarse_terrain.fields[field_name]):
                        merged_terrain.fields[field_name][fine_idx] = \
                            coarse_terrain.fields[field_name][coarse_idx]
        
        # If coarse terrain has computed weather, we can use it directly
        # Otherwise compute fresh
        if 'temperature' not in merged_terrain.fields or 'precipitation' not in merged_terrain.fields:
            merged_terrain.compute_weather()
    
    # Step 9: Build fine_regions for watershed projection
    fine_regions = {}
    for fine_idx in range(len(merged_terrain.elevations)):
        if fine_idx in fine_grid.invalidRegion:
            continue
        coarse_idx = mapper(fine_idx)
        if coarse_idx >= 0:
            if coarse_idx not in fine_regions:
                fine_regions[coarse_idx] = set()
            fine_regions[coarse_idx].add(fine_idx)
    
    # Step 10: Project watersheds
    zoomed_sheds = self.project_watersheds(self.basin, fine_regions, merged_terrain)
    
    zoomed_basins = DrainageBasins(merged_terrain, compute=False)
    zoomed_basins.sheds = zoomed_sheds
    
    return ZoomResult(
        terrain=merged_terrain,
        basins=zoomed_basins,
        mapper=mapper,
        chunks_loaded=len(chunk_offsets)
    )


Can you build a demo for zoom_region_fast

## Bug fizing

@patch
def _load_chunk_as_proxy(self: GeoStorage, 
                         cover_id: int,
                         origin_pos: HexPosition,
                         scale: int) -> LoadResult:
    """Load cached chunk as DataProxy (fully numpy-optimized)."""
    try:
        world = self.worlds[cover_id]
        cover_data = GeoStorage._get(world, 'cover_data')
        if not cover_data:
            return LoadResult(None, 'error', 'No cover data found')
        
        original_cover = ChunkCover.decode(cover_data)
        base_radius = original_cover.terrain.hexGrid.radius
        radius = base_radius / scale
        
        # Hardcode column order to avoid cursor.description issue
        cols = ['id', 'world_id', 'q', 'r', 's', 'grid_index', 'elevation',
                'plate_id', 'latitude', 'longitude', 'distance_from_coast',
                'watershed_id', 'chunk_q', 'chunk_r', 'chunk_s',
                'scale_level', 'modified', 'last_accessed', 'access_count']
        
        raw_rows = self.db.execute("""
            SELECT h.* FROM hex_data h
            INNER JOIN (
                SELECT world_id, q, r, s, MAX(modified) as max_mod
                FROM hex_data
                WHERE world_id = ? 
                  AND chunk_q = ? AND chunk_r = ? AND chunk_s = ?
                  AND scale_level = ?
                GROUP BY world_id, q, r, s
            ) latest 
                ON h.world_id = latest.world_id 
                AND h.q = latest.q AND h.r = latest.r AND h.s = latest.s
                AND h.modified = latest.max_mod
        """, [cover_id, origin_pos.q, origin_pos.r, origin_pos.s, scale]).fetchall()
        
        if not raw_rows:
            return LoadResult(None, 'not_found', 'No cached chunk')
        
        rows = [dict(zip(cols, row)) for row in raw_rows]
        
        # Fire-and-forget access tracking
        now = int(datetime.now().timestamp())
        self.db.execute("""
            UPDATE hex_data 
            SET last_accessed = ?, access_count = access_count + 1
            WHERE world_id = ? AND chunk_q = ? AND chunk_r = ? AND chunk_s = ?
            AND scale_level = ?
        """, [now, cover_id, origin_pos.q, origin_pos.r, origin_pos.s, scale])
        
        # --- Pure numpy from here ---
        
        qs = np.array([r['q'] for r in rows])
        rs = np.array([r['r'] for r in rows])
        ss = np.array([r['s'] for r in rows])
        
        min_q, max_q = int(qs.min()), int(qs.max())
        min_r, max_r = int(rs.min()), int(rs.max())
        
        nRows = max_r - min_r + 1
        nCols = max_q - min_q + 1
        
        elevations = np.full(nRows * nCols, -100.0)
        watersheds = np.full(nRows * nCols, -1, dtype=int)
        
        # Check if stored grid_index values are usable
        has_grid_index = all(
            row['grid_index'] is not None 
            and 0 <= row['grid_index'] < nRows * nCols
            for row in rows
        )
        
        if has_grid_index:
            indices = np.array([row['grid_index'] for row in rows], dtype=int)
        else:
            # Vectorized coordinate conversion
            positions = np.column_stack([
                qs - min_q,
                rs - min_r,
                ss - (-min_q - min_r)
            ])
            temp_grid = HexGrid(
                nRows=nRows, nCols=nCols, radius=radius,
                style=StyleCSS("simple")
            )
            indices = temp_grid.hexpositions_to_indices(positions, origin_index=0)
        
        # Extract data as numpy arrays
        elev_values = np.array([row['elevation'] for row in rows])
        ws_values = np.array([
            row['watershed_id'] if row['watershed_id'] is not None else -1
            for row in rows
        ], dtype=int)
        
        # Filter valid indices
        valid_mask = (indices >= 0) & (indices < nRows * nCols)
        valid_indices = indices[valid_mask]
        valid_elevs = elev_values[valid_mask]
        valid_ws = ws_values[valid_mask]
        
        # Vectorized assignment
        elevations[valid_indices] = valid_elevs
        watersheds[valid_indices] = valid_ws
        
        # Mark invalid hexes
        water_mask = valid_elevs <= -99.0
        invalidRegion = set(valid_indices[water_mask].tolist())
        
        actual_count = int(np.sum(elevations > -99.0))
        chunk_rings = original_cover.rings
        expected_count = 3 * chunk_rings * chunk_rings
        
        proxy = DataProxy(
            nRows=nRows, nCols=nCols, radius=radius,
            elevations=elevations, watersheds=watersheds,
            chunk_position=origin_pos, invalidRegion=invalidRegion,
            expected_count=expected_count, actual_count=actual_count
        )
        
        if not proxy.is_complete:
            return LoadResult(proxy, 'partial',
                            f'{actual_count}/{expected_count} hexes (incomplete)')
        
        return LoadResult(proxy, 'loaded', f'{len(rows)} hexes as DataProxy')
    
    except Exception as e:
        return LoadResult(None, 'error', str(e))


def stitch_chunks_fast(chunks: set[ChunkRef], 
                       cover: ChunkCover,
                       storage: GeoStorage, 
                       scale: int) -> tuple[DataProxy, dict[tuple, tuple], int, int]:
    """Load chunks as DataProxy and stitch using numpy block copies.
    
    Generates chunks on cache miss, then loads as lightweight proxy.
    Returns (stitched_proxy, chunk_offsets, chunk_rows, chunk_cols).
    """
    if not chunks:
        return None, {}, 0, 0
    
    coarse_grid = cover.terrain.hexGrid
    
    # Load (or generate) all chunks as DataProxy
    chunk_proxies = {}
    for chunk in chunks:
        origin_idx = coarse_grid.hexposition_to_index(chunk.center_hex, origin_index=0)
        if origin_idx < 0:
            continue  # Chunk center outside grid, skip
        
        # Use world-space position (matches what _save_chunk_cache stores)
        origin_pos = coarse_grid.index_to_hexposition(origin_idx)
        
        # Try loading from cache first
        result = storage._load_chunk_as_proxy(cover.ident, origin_pos, scale)
        
        if result.status == 'not_found':
            # Cache miss — generate the chunk, then load as proxy
            gen = storage.load_or_generate_chunk(cover, origin=origin_idx, scale=scale)
            if gen.status in ('generated', 'cached'):
                result = storage._load_chunk_as_proxy(cover.ident, origin_pos, scale)
            else:
                print(f"⚠️  Failed to generate chunk {chunk.key}: {gen.context}")
                continue
        
        if result.status == 'error':
            print(f"⚠️  Error loading chunk {chunk.key}: {result.context}")
            continue
        
        if result.status in ('loaded', 'partial'):
            coarse_row, coarse_col = coarse_grid.index_to_row_col(origin_idx)
            chunk_proxies[chunk.key] = (result.data, coarse_row, coarse_col)
    
    if not chunk_proxies:
        return None, {}, 0, 0
    
    # Validate dimensions — all chunks should be same size
    first_proxy = next(iter(chunk_proxies.values()))[0]
    chunk_rows = first_proxy.nRows
    chunk_cols = first_proxy.nCols
    radius = first_proxy.radius
    
    mismatched = {k for k, (p, _, _) in chunk_proxies.items() 
                  if p.nRows != chunk_rows or p.nCols != chunk_cols}
    if mismatched:
        print(f"⚠️  Dimension mismatch in chunks: {mismatched}, skipping them")
        for k in mismatched:
            del chunk_proxies[k]
    
    if not chunk_proxies:
        return None, {}, 0, 0
    
    # Calculate merged dimensions from coarse positions
    coarse_rows = [d[1] for d in chunk_proxies.values()]
    coarse_cols = [d[2] for d in chunk_proxies.values()]
    min_coarse_row, max_coarse_row = min(coarse_rows), max(coarse_rows)
    min_coarse_col, max_coarse_col = min(coarse_cols), max(coarse_cols)
    
    merged_rows = (max_coarse_row - min_coarse_row) * scale + chunk_rows
    merged_cols = (max_coarse_col - min_coarse_col) * scale + chunk_cols
    
    # Allocate merged 2D arrays
    merged_elev = np.full((merged_rows, merged_cols), -100.0)
    merged_ws = np.full((merged_rows, merged_cols), -1, dtype=int)
    merged_invalid = set()
    
    chunk_offsets = {}
    
    # Block-copy each chunk
    for key, (proxy, coarse_row, coarse_col) in chunk_proxies.items():
        row_off = (coarse_row - min_coarse_row) * scale
        col_off = (coarse_col - min_coarse_col) * scale
        chunk_offsets[key] = (row_off, col_off)
        
        # Reshape proxy to 2D
        p_elev = proxy.elevations.reshape(proxy.nRows, proxy.nCols)
        p_ws = proxy.watersheds.reshape(proxy.nRows, proxy.nCols)
        
        # Clip source/dest ranges
        sr0, sr1 = 0, proxy.nRows
        sc0, sc1 = 0, proxy.nCols
        dr0, dr1 = row_off, row_off + proxy.nRows
        dc0, dc1 = col_off, col_off + proxy.nCols
        
        if dr0 < 0:          sr0 -= dr0; dr0 = 0
        if dr1 > merged_rows: sr1 -= (dr1 - merged_rows); dr1 = merged_rows
        if dc0 < 0:          sc0 -= dc0; dc0 = 0
        if dc1 > merged_cols: sc1 -= (dc1 - merged_cols); dc1 = merged_cols
        
        if dr0 < dr1 and dc0 < dc1:
            merged_elev[dr0:dr1, dc0:dc1] = p_elev[sr0:sr1, sc0:sc1]
            merged_ws[dr0:dr1, dc0:dc1] = p_ws[sr0:sr1, sc0:sc1]
        
        # Remap invalidRegion with offset
        for local_idx in proxy.invalidRegion:
            lr = local_idx // proxy.nCols
            lc = local_idx % proxy.nCols
            mr = row_off + lr
            mc = col_off + lc
            if 0 <= mr < merged_rows and 0 <= mc < merged_cols:
                merged_invalid.add(mr * merged_cols + mc)
    
    merged_proxy = DataProxy(
        nRows=merged_rows,
        nCols=merged_cols,
        radius=radius,
        elevations=merged_elev.ravel(),
        watersheds=merged_ws.ravel(),
        invalidRegion=merged_invalid,
        expected_count=sum(p[0].expected_count for p in chunk_proxies.values()),
        actual_count=sum(p[0].actual_count for p in chunk_proxies.values())
    )
    
    return merged_proxy, chunk_offsets, chunk_rows, chunk_cols


## once more with feeling

In [ ]:
#| export
@dataclass
class DataProxy:
    """Lightweight chunk data holder for efficient stitching.
    
    Holds only raw numpy arrays without HexGrid geometry.
    """
    nRows: int
    nCols: int
    radius: float
    elevations: np.ndarray  # Shape: (nRows * nCols,)
    watersheds: Optional[np.ndarray] = None  # Shape: (nRows * nCols,), watershed IDs
    chunk_position: Optional[HexPosition] = None  # For debugging/tracking
    invalidRegion: set[int] = field(default_factory=set)  # Track invalid hexes
    
    # Metadata for validation
    expected_count: int = 0  # How many valid hexes we expect
    actual_count: int = 0    # How many we actually loaded
    
    @property
    def size(self) -> int:
        return self.nRows * self.nCols
    
    def __len__(self):
        return self.size
    
    @property
    def is_complete(self) -> bool:
        """Check if all expected hexes were loaded."""
        if self.expected_count == 0:
            return True  # No expectation set
        return self.actual_count >= self.expected_count * 0.95  # Allow 5% tolerance
    
    def to_dict(self) -> dict:
        """Convert to dict for debugging."""
        return {
            'nRows': self.nRows,
            'nCols': self.nCols,
            'radius': self.radius,
            'size': self.size,
            'has_watersheds': self.watersheds is not None,
            'chunk_position': str(self.chunk_position) if self.chunk_position else None,
            'invalid_count': len(self.invalidRegion),
            'complete': self.is_complete,
            'expected': self.expected_count,
            'actual': self.actual_count
        }

In [ ]:
#| export
@patch
def _load_chunk_as_proxy(self: GeoStorage, 
                         cover_id: int,
                         origin_pos: HexPosition,
                         scale: int) -> LoadResult:
    """Load cached chunk as DataProxy (fully numpy-optimized)."""
    try:
        # Step 1: Get chunk metadata — no cover decode needed
        meta = list(self.chunk_meta.rows_where(
            "world_id = ? AND chunk_q = ? AND chunk_r = ? AND chunk_s = ? AND scale_level = ?",
            [cover_id, origin_pos.q, origin_pos.r, origin_pos.s, scale]
        ))
        
        if not meta:
            return LoadResult(None, 'not_found', 'No chunk metadata')
        
        meta = meta[0]
        nRows = meta['nRows']
        nCols = meta['nCols']
        radius = meta['radius']
        expected_count = meta['hex_count']
        
        # Step 2: Load hex data (minimal columns only)
        raw_rows = self.db.execute("""
            SELECT grid_index, elevation, watershed_id
            FROM hex_data
            WHERE world_id = ? 
              AND chunk_q = ? AND chunk_r = ? AND chunk_s = ?
              AND scale_level = ?
        """, [cover_id, origin_pos.q, origin_pos.r, origin_pos.s, scale]).fetchall()
        
        if not raw_rows:
            return LoadResult(None, 'not_found', 'No cached hex data')
        
        # Step 3: Update access stats on chunk_meta (fire-and-forget)
        now = int(datetime.now().timestamp())
        self.chunk_meta.update({
            'id': meta['id'],
            'last_accessed': now,
            'access_count': meta['access_count'] + 1
        })
        
        # Step 4: Pure numpy — no coordinate conversion needed
        grid_size = nRows * nCols
        elevations = np.full(grid_size, -100.0)
        watersheds = np.full(grid_size, -1, dtype=int)
        
        # Extract columns as numpy arrays
        indices = np.array([row[0] for row in raw_rows], dtype=int)
        elev_values = np.array([row[1] for row in raw_rows])
        ws_values = np.array([row[2] if row[2] is not None else -1 
                              for row in raw_rows], dtype=int)
        
        # Filter valid indices
        valid_mask = (indices >= 0) & (indices < grid_size)
        valid_indices = indices[valid_mask]
        
        # Vectorized assignment
        elevations[valid_indices] = elev_values[valid_mask]
        watersheds[valid_indices] = ws_values[valid_mask]
        
        # Mark invalid hexes (sentinel elevation)
        water_mask = elev_values[valid_mask] <= -99.0
        invalidRegion = set(valid_indices[water_mask].tolist())
        
        actual_count = int(np.sum(elevations > -99.0))
        
        proxy = DataProxy(
            nRows=nRows, nCols=nCols, radius=radius,
            elevations=elevations, watersheds=watersheds,
            chunk_position=origin_pos, invalidRegion=invalidRegion,
            expected_count=expected_count, actual_count=actual_count
        )
        
        if not proxy.is_complete:
            return LoadResult(proxy, 'partial',
                            f'{actual_count}/{expected_count} hexes (incomplete)')
        
        return LoadResult(proxy, 'loaded', f'{len(raw_rows)} hexes as DataProxy')
    
    except Exception as e:
        return LoadResult(None, 'error', str(e))


In [ ]:
#| export




def stitch_chunks_fast(chunks: set[ChunkRef], 
                       cover: ChunkCover,
                       storage: GeoStorage, 
                       scale: int) -> tuple[DataProxy, dict[tuple, tuple], int, int]:
    """Load chunks as DataProxy and stitch using numpy block copies.
    
    Generates chunks on cache miss, then loads as lightweight proxy.
    Returns (stitched_proxy, chunk_offsets, chunk_rows, chunk_cols).
    """
    if not chunks:
        return None, {}, 0, 0
    
    coarse_grid = cover.terrain.hexGrid
    
    # Load (or generate) all chunks as DataProxy
    chunk_proxies = {}
    for chunk in chunks:
        origin_idx = coarse_grid.hexposition_to_index(chunk.center_hex, origin_index=0)
        if origin_idx < 0:
            continue
        
        # Use world-space position (matches what _save_chunk_cache stores)
        origin_pos = coarse_grid.index_to_hexposition(origin_idx)
        
        # Try loading from cache first
        result = storage._load_chunk_as_proxy(cover.ident, origin_pos, scale)
        
        if result.status == 'not_found':
            # Cache miss — generate, then load as proxy
            gen = storage.load_or_generate_chunk(cover, origin=origin_idx, scale=scale)
            if gen.status in ('generated', 'cached'):
                result = storage._load_chunk_as_proxy(cover.ident, origin_pos, scale)
            else:
                print(f"⚠️  Failed to generate chunk {chunk.key}: {gen.context}")
                continue
        
        if result.status == 'error':
            print(f"⚠️  Error loading chunk {chunk.key}: {result.context}")
            continue
        
        if result.status in ('loaded', 'partial'):
            coarse_row, coarse_col = coarse_grid.index_to_row_col(origin_idx)
            chunk_proxies[chunk.key] = (result.data, coarse_row, coarse_col)
    
    if not chunk_proxies:
        return None, {}, 0, 0
    
    # Validate dimensions — all chunks should be same size
    first_proxy = next(iter(chunk_proxies.values()))[0]
    chunk_rows = first_proxy.nRows
    chunk_cols = first_proxy.nCols
    radius = first_proxy.radius
    
    mismatched = {k for k, (p, _, _) in chunk_proxies.items() 
                  if p.nRows != chunk_rows or p.nCols != chunk_cols}
    if mismatched:
        print(f"⚠️  Dimension mismatch in chunks: {mismatched}, skipping them")
        for k in mismatched:
            del chunk_proxies[k]
    
    if not chunk_proxies:
        return None, {}, 0, 0
    
    # Calculate merged dimensions from coarse positions
    coarse_rows = [d[1] for d in chunk_proxies.values()]
    coarse_cols = [d[2] for d in chunk_proxies.values()]
    min_coarse_row, max_coarse_row = min(coarse_rows), max(coarse_rows)
    min_coarse_col, max_coarse_col = min(coarse_cols), max(coarse_cols)
    
    merged_rows = (max_coarse_row - min_coarse_row) * scale + chunk_rows
    merged_cols = (max_coarse_col - min_coarse_col) * scale + chunk_cols
    
    # Allocate merged 2D arrays
    merged_elev = np.full((merged_rows, merged_cols), -100.0)
    merged_ws = np.full((merged_rows, merged_cols), -1, dtype=int)
    merged_invalid = set()
    
    chunk_offsets = {}
    
    # Block-copy each chunk using numpy 2D slicing
    for key, (proxy, coarse_row, coarse_col) in chunk_proxies.items():
        row_off = (coarse_row - min_coarse_row) * scale
        col_off = (coarse_col - min_coarse_col) * scale
        chunk_offsets[key] = (row_off, col_off)
        
        # Reshape proxy to 2D for block copy
        p_elev = proxy.elevations.reshape(proxy.nRows, proxy.nCols)
        p_ws = proxy.watersheds.reshape(proxy.nRows, proxy.nCols)
        
        # Source and dest ranges
        sr0, sr1 = 0, proxy.nRows
        sc0, sc1 = 0, proxy.nCols
        dr0, dr1 = row_off, row_off + proxy.nRows
        dc0, dc1 = col_off, col_off + proxy.nCols
        
        # Clip to merged bounds
        if dr0 < 0:           sr0 -= dr0; dr0 = 0
        if dr1 > merged_rows: sr1 -= (dr1 - merged_rows); dr1 = merged_rows
        if dc0 < 0:           sc0 -= dc0; dc0 = 0
        if dc1 > merged_cols: sc1 -= (dc1 - merged_cols); dc1 = merged_cols
        
        # Single numpy block copy per chunk
        if dr0 < dr1 and dc0 < dc1:
            merged_elev[dr0:dr1, dc0:dc1] = p_elev[sr0:sr1, sc0:sc1]
            merged_ws[dr0:dr1, dc0:dc1] = p_ws[sr0:sr1, sc0:sc1]
        
        # Remap invalidRegion with offset
        for local_idx in proxy.invalidRegion:
            lr = local_idx // proxy.nCols
            lc = local_idx % proxy.nCols
            mr = row_off + lr
            mc = col_off + lc
            if 0 <= mr < merged_rows and 0 <= mc < merged_cols:
                merged_invalid.add(mr * merged_cols + mc)
    
    merged_proxy = DataProxy(
        nRows=merged_rows,
        nCols=merged_cols,
        radius=radius,
        elevations=merged_elev.ravel(),
        watersheds=merged_ws.ravel(),
        invalidRegion=merged_invalid,
        expected_count=sum(p[0].expected_count for p in chunk_proxies.values()),
        actual_count=sum(p[0].actual_count for p in chunk_proxies.values())
    )
    
    return merged_proxy, chunk_offsets, chunk_rows, chunk_cols



def proxy_to_terrain(proxy: DataProxy, 
                     climate: ClimatePreset = None,
                     geo: GeoBounds = None) -> Terrain:
    """Convert DataProxy to full Terrain with geometry.
    
    Builds HexGrid once on the final merged result.
    """
    bounds = MapRect(
        MapCord(0, 0), 
        MapSize(proxy.nRows * proxy.radius, proxy.nCols * proxy.radius)
    )
    
    terrain = Terrain(
        bounds=bounds, 
        radius=proxy.radius,
        climate=climate,
        geo=geo
    )
    
    terrain.hexGrid.nRows = proxy.nRows
    terrain.hexGrid.nCols = proxy.nCols
    terrain.hexGrid.adjustRadius(proxy.radius)
    
    terrain.elevations = proxy.elevations.copy()
    
    if proxy.watersheds is not None and np.any(proxy.watersheds >= 0):
        terrain.fields['watershed_id'] = proxy.watersheds.copy()
    
    terrain.hexGrid.invalidRegion = proxy.invalidRegion.copy()
    
    return terrain



@patch
def zoom_region_fast(self: ChunkCover, 
                     region: HexRegion,
                     scale: int = 2,
                     compute_weather: bool = True) -> ZoomResult:
    """Zoom into a region using DataProxy for efficient stitching."""
    if self.db is None:
        raise ValueError("ChunkCover.db must be set")
    
    # Step 1: Bounding box with padding
    min_row, max_row, min_col, max_col = region_bounding_box(region)
    
    padding = self.halo_rings + 1
    min_row = max(0, min_row - padding)
    max_row = min(self.terrain.hexGrid.nRows - 1, max_row + padding)
    min_col = max(0, min_col - padding)
    max_col = min(self.terrain.hexGrid.nCols - 1, max_col + padding)
    
    # Step 2: Find chunks
    chunks = bbox_to_chunk_refs(min_row, max_row, min_col, max_col, self)
    
    # Step 3: Load and stitch as DataProxy
    merged_proxy, chunk_offsets, chunk_rows, chunk_cols = stitch_chunks_fast(
        chunks, self, self.db, scale
    )
    
    if merged_proxy is None:
        return ZoomResult(None, None, lambda x: -1, 0)
    
    if not merged_proxy.is_complete:
        print(f"⚠️  Merged terrain incomplete: {merged_proxy.actual_count}/{merged_proxy.expected_count} hexes")
    
    # Step 4: Build Terrain once from merged proxy
    merged_terrain = proxy_to_terrain(
        merged_proxy,
        climate=getattr(self.terrain, 'climate', None),
        geo=getattr(self.terrain, 'geo', None)
    )
    
    fine_grid = merged_terrain.hexGrid
    
    # Step 5: Lazy init coarse basin
    if self.basin is None:
        self.basin = DrainageBasins(self.terrain)
        self.save()
    
    # Step 6: Build mapper
    mapper = build_fine_to_coarse_mapper(
        self, merged_terrain, chunk_offsets, scale,
        chunk_rows=chunk_rows, chunk_cols=chunk_cols
    )
    
    # Step 7: Extend invalidRegion using mapper
    for fine_idx in range(len(merged_terrain.elevations)):
        if fine_idx in fine_grid.invalidRegion:
            continue
        coarse_idx = mapper(fine_idx)
        if coarse_idx < 0:
            fine_grid.invalidRegion.add(fine_idx)
        elif merged_terrain.elevations[fine_idx] <= -99.0:
            fine_grid.invalidRegion.add(fine_idx)
    
    # Step 8: Project weather fields from coarse → fine
    if compute_weather:
        coarse_terrain = self.terrain
        weather_fields = ['temperature', 'precipitation', 'humidity',
                         'climate_pet', 'aridity_index', 'climate',
                         'latitude', 'longitude', 'distance_to_coast',
                         'precip_rate_mmh']
        
        # Build mapping arrays once (reuse for all fields)
        fine_indices = []
        coarse_indices = []
        for fine_idx in range(len(merged_terrain.elevations)):
            if fine_idx in fine_grid.invalidRegion:
                continue
            coarse_idx = mapper(fine_idx)
            if coarse_idx >= 0:
                fine_indices.append(fine_idx)
                coarse_indices.append(coarse_idx)
        
        fine_arr = np.array(fine_indices, dtype=int)
        coarse_arr = np.array(coarse_indices, dtype=int)
        
        for field_name in weather_fields:
            if field_name in coarse_terrain.fields:
                coarse_field = coarse_terrain.fields[field_name]
                # Clip coarse indices to valid range
                valid = coarse_arr < len(coarse_field)
                
                merged_terrain.fields[field_name] = np.zeros(len(merged_terrain.elevations))
                merged_terrain.fields[field_name][fine_arr[valid]] = coarse_field[coarse_arr[valid]]
        
        # Fallback: compute fresh if coarse had no weather
        if 'temperature' not in merged_terrain.fields or 'precipitation' not in merged_terrain.fields:
            merged_terrain.compute_weather()
    
    # Step 9: Build fine_regions for watershed projection
    fine_regions = {}
    for fi, ci in zip(fine_indices, coarse_indices):
        if ci not in fine_regions:
            fine_regions[ci] = set()
        fine_regions[ci].add(fi)
    
    # Step 10: Project watersheds
    zoomed_sheds = self.project_watersheds(self.basin, fine_regions, merged_terrain)
    
    zoomed_basins = DrainageBasins(merged_terrain, compute=False)
    zoomed_basins.sheds = zoomed_sheds
    
    return ZoomResult(
        terrain=merged_terrain,
        basins=zoomed_basins,
        mapper=mapper,
        chunks_loaded=len(chunk_offsets)
    )



In [ ]:
# Demo: zoom_region_fast with weather projection
with GeoStorageDebugger(keep_on_error=True) as dbg:
    # 1. Load terrain and compute weather on coarse
    terrain = TerraDemo().bayArea_map()
    terrain.compute_weather()
    
    print(f"Coarse terrain: {terrain.hexGrid.nRows}x{terrain.hexGrid.nCols} = {len(terrain.hexGrid.hexes)} hexes")
    print(f"Weather fields: {list(terrain.fields.keys())}")
    
    # 2. Create cover and attach DB
    cover = ChunkCover(terrain, rings=5, halo_rings=1)
    cover.db = dbg.server
    cover.save(name="Bay Area Fast Test")
    
    # 3. Compute basins on coarse terrain (lazy init will happen in zoom)
    basins = DrainageBasins(terrain)
    cover.basin = basins
    
    print(f"Found {len(basins.sheds)} watersheds")
    
    # 4. Pick a medium-sized watershed
    shed_sizes = [(i, len(ws.region.hexes)) for i, ws in enumerate(basins.sheds)]
    shed_sizes.sort(key=lambda x: x[1], reverse=True)
    
    target_idx = shed_sizes[2][0] if len(shed_sizes) > 2 else 0
    target_shed = basins.sheds[target_idx]
    region = target_shed.region
    
    print(f"\nSelected watershed {target_idx}: {len(region.hexes)} hexes")
    
    # 5. Zoom using fast path
    import time
    start = time.time()
    result = cover.zoom_region_fast(region, scale=2, compute_weather=True)
    elapsed = time.time() - start
    
    print(f"\n✓ Zoom completed in {elapsed:.2f}s")
    print(f"  Chunks loaded: {result.chunks_loaded}")
    print(f"  Fine terrain: {result.terrain.hexGrid.nRows}x{result.terrain.hexGrid.nCols}")
    print(f"  Invalid hexes: {len(result.terrain.hexGrid.invalidRegion)}")
    print(f"  Watersheds: {len(result.basins.sheds)}")
    
    # 6. Verify weather was projected
    fine_terrain = result.terrain
    print(f"\nFine terrain fields: {list(fine_terrain.fields.keys())}")
    
    # Check a few hexes have valid weather data
    valid_count = 0
    for idx in range(min(100, len(fine_terrain.elevations))):
        if idx not in fine_terrain.hexGrid.invalidRegion:
            if 'temperature' in fine_terrain.fields and fine_terrain.fields['temperature'][idx] != 0:
                valid_count += 1
    
    print(f"Sample check: {valid_count}/100 hexes have projected temperature")
    
    # 7. Visualize
    fine_terrain.colorMap()
    
    # Optionally add climate overlay
    if 'climate' in fine_terrain.fields:
        fine_terrain.add_climate_overlay()
    
    fine_terrain.hexGrid.update()
    
    print("\n✓ Visualization ready")

# Show the result
result.terrain.hexGrid.builder.show()


In [ ]:
# Quick round-trip test: generate → load as proxy → stitch → terrain
with GeoStorageDebugger(keep_on_error=True) as dbg:
    terrain = TerraDemo().bayArea_map()
    terrain.compute_weather()
    
    cover = ChunkCover(terrain, rings=5, halo_rings=1)
    cover.db = dbg.server
    cover.save(name="Bay Area Proxy Test")
    cover.basin = DrainageBasins(terrain)
    
    # Pick a watershed
    shed_sizes = [(i, len(ws.region.hexes)) for i, ws in enumerate(cover.basin.sheds)]
    shed_sizes.sort(key=lambda x: x[1], reverse=True)
    region = cover.basin.sheds[shed_sizes[2][0]].region
    
    # Test 1: Verify proxy loading works with world-space coords
    coarse_grid = cover.terrain.hexGrid
    chunks = bbox_to_chunk_refs(*region_bounding_box(region), cover)
    for chunk in chunks:
        idx = coarse_grid.hexposition_to_index(chunk.center_hex, origin_index=0)
        if idx < 0: continue
        
        # Generate chunk
        gen = dbg.server.load_or_generate_chunk(cover, origin=idx, scale=2)
        print(f"Generate {chunk.key}: {gen.status}")
        
        # Load as proxy with WORLD-SPACE coords
        origin_pos = coarse_grid.index_to_hexposition(idx)
        proxy = dbg.server._load_chunk_as_proxy(cover.ident, origin_pos, 2)
        print(f"Proxy load: {proxy.status} - {proxy.context}")
        if proxy.data:
            print(f"  Proxy: {proxy.data.nRows}x{proxy.data.nCols}, {proxy.data.actual_count} valid hexes")
    
    # Test 2: Full zoom_region_fast
    import time
    t0 = time.time()
    result = cover.zoom_region_fast(region, scale=2, compute_weather=True)
    elapsed = time.time() - t0
    
    print(f"\n--- zoom_region_fast ---")
    print(f"Time: {elapsed:.2f}s")
    print(f"Chunks: {result.chunks_loaded}")
    print(f"Terrain: {result.terrain.hexGrid.nRows}x{result.terrain.hexGrid.nCols}" if result.terrain else "None!")
    print(f"Invalid: {len(result.terrain.hexGrid.invalidRegion)}" if result.terrain else "")
    print(f"Watersheds: {len(result.basins.sheds)}" if result.basins else "")
    print(f"Fields: {list(result.terrain.fields.keys())}" if result.terrain else "")
    
    # Test 3: Verify weather projection has actual values
    if result.terrain and 'temperature' in result.terrain.fields:
        temps = result.terrain.fields['temperature']
        valid_mask = np.array([i not in result.terrain.hexGrid.invalidRegion 
                               for i in range(len(temps))])
        valid_temps = temps[valid_mask]
        print(f"Temperature: min={valid_temps.min():.1f}, max={valid_temps.max():.1f}, mean={valid_temps.mean():.1f}")


In [ ]:
SVGBuilder.BUILDERHIDE = False

In [ ]:
with GeoStorageDebugger(keep_on_error=True) as dbg:
    # 1. Load and prep terrain
    terrain = TerraDemo().bayArea_map()
    terrain.compute_weather()
    terrain.carve_to_ocean(num_lakes=1)
    
    # 2. Create cover with DB
    cover = ChunkCover(terrain, rings=5, halo_rings=1)
    cover.db = dbg.server
    cover.save(name="Bay Area")
    
    # 3. Compute basins and find 3rd largest watershed
    cover.basin = DrainageBasins(terrain)
    shed_sizes = [(i, len(ws.region.hexes)) for i, ws in enumerate(cover.basin.sheds)]
    shed_sizes.sort(key=lambda x: x[1], reverse=True)
    
    target_idx = shed_sizes[2][0]  # 3rd largest
    region = cover.basin.sheds[target_idx].region
    print(f"3rd largest watershed: index={target_idx}, {len(region.hexes)} hexes")
    
    # 4. Zoom into that watershed
    result = cover.zoom_region_fast(region, scale=2, compute_weather=True)
    
    print(f"Terrain: {result.terrain.hexGrid.nRows}x{result.terrain.hexGrid.nCols}")
    print(f"Chunks: {result.chunks_loaded}, Watersheds: {len(result.basins.sheds)}")
    print(f"Fields: {list(result.terrain.fields.keys())}")
    
    # 5. Visualize
    result.terrain.hexGrid.adjustRadius(terrain.hexGrid.radius)
    result.terrain.colorMap()
    result.terrain.hexGrid.update()

result.terrain.hexGrid.builder.show()


an you for the above demo find the third largest watershed and zoom

# Compare zoom_region vs zoom_region_fast
with GeoStorageDebugger(keep_on_error=True) as dbg:
    terrain = TerraDemo().bayArea_map()
    terrain.compute_weather()
    
    # Setup two covers pointing at same DB
    cover_old = ChunkCover(terrain, rings=5, halo_rings=1)
    cover_old.db = dbg.server
    cover_old.save(name="Compare Old")
    cover_old.basin = DrainageBasins(terrain)
    
    cover_fast = ChunkCover(terrain, rings=5, halo_rings=1)
    cover_fast.db = dbg.server
    cover_fast.ident = cover_old.ident  # Same world
    cover_fast.basin = cover_old.basin
    
    # Pick a watershed
    shed_sizes = [(i, len(ws.region.hexes)) for i, ws in enumerate(cover_old.basin.sheds)]
    shed_sizes.sort(key=lambda x: x[1], reverse=True)
    region = cover_old.basin.sheds[shed_sizes[2][0]].region
    print(f"Region: {len(region.hexes)} hexes\n")
    
    # Run both
    import time
    
    t0 = time.time()
    result_old = cover_old.zoom_region(region, scale=2, compute_weather=True)
    t_old = time.time() - t0
    
    t0 = time.time()
    result_fast = cover_fast.zoom_region_fast(region, scale=2, compute_weather=True)
    t_fast = time.time() - t0
    
    print(f"{'':20s} {'Old':>12s} {'Fast':>12s}")
    print(f"{'─'*46}")
    print(f"{'Time (s)':20s} {t_old:12.2f} {t_fast:12.2f}")
    print(f"{'Grid size':20s} {str(result_old.terrain.hexGrid.nRows)+'x'+str(result_old.terrain.hexGrid.nCols):>12s} {str(result_fast.terrain.hexGrid.nRows)+'x'+str(result_fast.terrain.hexGrid.nCols):>12s}")
    print(f"{'Total hexes':20s} {len(result_old.terrain.elevations):12d} {len(result_fast.terrain.elevations):12d}")
    print(f"{'Invalid hexes':20s} {len(result_old.terrain.hexGrid.invalidRegion):12d} {len(result_fast.terrain.hexGrid.invalidRegion):12d}")
    print(f"{'Chunks loaded':20s} {result_old.chunks_loaded:12d} {result_fast.chunks_loaded:12d}")
    print(f"{'Watersheds':20s} {len(result_old.basins.sheds):12d} {len(result_fast.basins.sheds):12d}")
    
    # Compare elevations
    e_old = result_old.terrain.elevations
    e_fast = result_fast.terrain.elevations
    
    if len(e_old) == len(e_fast):
        # Only compare valid hexes
        valid_old = set(range(len(e_old))) - result_old.terrain.hexGrid.invalidRegion
        valid_fast = set(range(len(e_fast))) - result_fast.terrain.hexGrid.invalidRegion
        valid_both = valid_old & valid_fast
        
        idx = np.array(sorted(valid_both))
        elev_match = np.allclose(e_old[idx], e_fast[idx], atol=1.0)
        elev_diff = np.abs(e_old[idx] - e_fast[idx])
        
        print(f"\n{'─'*46}")
        print(f"{'Valid hex overlap':20s} {len(valid_both):12d}")
        print(f"{'Elevation match':20s} {'✓' if elev_match else '✗':>12s}")
        print(f"{'Max elev diff':20s} {elev_diff.max():12.2f}")
        print(f"{'Mean elev diff':20s} {elev_diff.mean():12.4f}")
    else:
        print(f"\n⚠️  Different array sizes: {len(e_old)} vs {len(e_fast)}")
    
    # Compare weather fields
    print(f"\n{'─'*46}")
    print(f"Fields comparison:")
    old_fields = set(result_old.terrain.fields.keys())
    fast_fields = set(result_fast.terrain.fields.keys())
    print(f"  Old only:  {old_fields - fast_fields or 'none'}")
    print(f"  Fast only: {fast_fields - old_fields or 'none'}")
    print(f"  Shared:    {old_fields & fast_fields}")
    
    for field_name in sorted(old_fields & fast_fields):
        f_old = result_old.terrain.fields[field_name]
        f_fast = result_fast.terrain.fields[field_name]
        if len(f_old) == len(f_fast) and len(idx) > 0:
            diff = np.abs(f_old[idx] - f_fast[idx])
            match = np.allclose(f_old[idx], f_fast[idx], atol=1.0)
            print(f"  {field_name:25s} max_diff={diff.max():.3f}  {'✓' if match else '✗'}")
    
    # Compare mapper outputs
    print(f"\n{'─'*46}")
    print(f"Mapper comparison (sample of 50):")
    sample_idx = sorted(valid_both)[:50]
    mapper_match = 0
    mapper_total = 0
    for fi in sample_idx:
        c_old = result_old.mapper(fi)
        c_fast = result_fast.mapper(fi)
        if c_old == c_fast:
            mapper_match += 1
        mapper_total += 1
    print(f"  {mapper_match}/{mapper_total} mapper outputs match")
    
    print(f"\n{'═'*46}")
    all_good = (elev_match if len(e_old) == len(e_fast) else False)
    print(f"{'PASS ✓' if all_good else 'DIFFERENCES FOUND ⚠️'}")
    print(f"Speedup: {t_old/t_fast:.1f}×")


Did things improve?

Yes, significantly! Comparing the two runs:

| Metric | Old (terrain) | New (cover) | Improvement |
|--------|--------------|-------------|-------------|
| **Total time** | **92s** | **21s** | **4.4× faster** |
| `deepcopy`/`combine_rivers` | 47.6s | **0s** ✅ | Eliminated |
| `_upsample_bilinear` | 12.5s | **0s** ✅ | Eliminated |
| `_save_chunk_cache` (DB writes) | 10.6s | **0s** ✅ | Eliminated |
| `_build_hexes` | 14.5s | 10.5s | Similar |
| `soil.from_plates` | 7.4s | 7.0s | Similar |
| `fetchall` (DB reads) | 2.3s | 2.3s | Same |

The O(n²) `combine_rivers` / 26M `deepcopy` calls are completely gone — that's the big win.

The remaining 21s is now dominated by two things:
1. **`_build_hexes` + `Hex.__init__` (~14s)** — building hex grids for all 20 chunks individually during decode, most of which get thrown away after stitching
2. **`soil.from_plates` (~7s)** — rebuilding watershed soil data inside `basin.decode` for each chunk

⚠️ One thing to watch: the latest run shows `"Dimension mismatch in chunks"` skipping 16 of 20 chunks — only 4 actually loaded. So it may be faster partly because it's doing less work (and the result might be incomplete).

The next optimization target is clear: a **lightweight chunk decode** that skips `_build_hexes` and `basin.decode` per-chunk, and only builds the grid + projects basins once on the final stitched terrain.

Now that I have zoom_region_fast what shoud I get rid of

Did I do it ok?

### crop to region


We have been rather generous around the region, but it would be good to crop to the region before returning the terrain. How would that work?